Data Quality: Initial Missing-Value Assessment & Duplicate Analysis


### 1. Dataset Loading

**Objective:** Load the original dataset to perform Data Quality Assessment.

The dataset is read from the raw data directory using pandas. Since the dataset contains millions of records, it is loaded in chunks of 100,000 rows to manage the reading process. The chunks are then combined into a single DataFrame for subsequent analysis.

**Note:** Combining all chunks still requires enough RAM to store the complete dataset.
  

In [2]:
import pandas as pd

DATA_PATH = "../data/raw/data.csv"
CHUNK_SIZE = 100_000

chunks = []

for chunk in pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)

print(f"Total rows: {len(df):,}")
print(f"Total columns: {df.shape[1]}")

Total rows: 8,714,356
Total columns: 20



## 2. Data Quality Assessment

### 2.1 Missing Values Analysis

**Objective:** Evaluate dataset completeness by identifying missing values in each attribute.

The analysis is performed on the complete dataset containing **8,714,356 records and 20 attributes**.

**Methodology:**
1. Verify that the complete dataset has been loaded correctly.
2. Count missing values for each attribute using `isnull().sum()`.
3. Calculate the percentage of missing values using `isnull().mean() * 100`.
4. Sort the attributes by missing-value percentage in descending order.

**Purpose:**

This analysis helps identify attributes with incomplete information and determine which columns may require further investigation or cleaning.

Missing values are not automatically removed because some attributes, such as reply and repost identifiers, may be naturally missing when a post is not a reply or repost.

**Outcome:** A summary table showing the number and percentage of missing values for each attribute.


In [3]:

# 2.1 Missing Values Analysis

# Verify dataset dimensions
total_rows = len(df)
total_columns = df.shape[1]

print("=== DATASET VERIFICATION ===")
print(f"Total rows: {total_rows:,}")
print(f"Total columns: {total_columns}")

assert total_rows == 8_714_356, "Dataset size mismatch!"
assert total_columns == 20, "Column count mismatch!"

print("Full dataset verified successfully!")

# Calculate missing values
missing_count = df.isnull().sum()
missing_percentage = df.isnull().mean() * 100

# Create summary table
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage (%)": missing_percentage.round(2)
})

# Sort by missing percentage
missing_summary = missing_summary.sort_values(
    by="Missing Percentage (%)",
    ascending=False
)

display(missing_summary)


=== DATASET VERIFICATION ===
Total rows: 8,714,356
Total columns: 20
Full dataset verified successfully!


,Missing Count,Missing Percentage (%)
in_reply_to_accountid,7839269,89.96
in_reply_to_postid,7828114,89.83
reposted_postid,5109181,58.63
reposted_accountid,5109181,58.63
account_profile_description,1669919,19.16
post_language,103743,1.19
follower_count,1318,0.02
following_count,1192,0.01
is_repost,0,0.00
account_mentions,0,0.00



### 2.2 Duplicate Records Analysis

**Objective:** Identify duplicated observations in the complete dataset.

The analysis distinguishes between:

- **Full duplicates:** Records with identical values across all attributes.
- **Repeated post IDs:** Records sharing the same post identifier, which may require further investigation.

This step measures the extent of duplication without modifying or removing any records.


In [4]:

# 2.2 Duplicate Records Analysis
# Identify duplicates in the complete dataset

print("=== DUPLICATE ANALYSIS ===")

# Step 1: Verify dataset size
total_rows = len(df)

assert total_rows == 8_714_356, (
    f"Unexpected dataset size: {total_rows:,}"
)

# Step 2: Count fully duplicated rows
full_duplicates = df.duplicated().sum()

# Step 3: Count repeated post IDs
repeated_postids = df.duplicated(
    subset=["postid"]
).sum()

# Step 4: Calculate duplicate percentages
full_duplicate_pct = full_duplicates / total_rows * 100
repeated_postid_pct = repeated_postids / total_rows * 100

# Step 5: Display results
print(f"Total rows: {total_rows:,}")
print(f"Fully duplicated rows: {full_duplicates:,}")
print(f"Repeated post ID occurrences: {repeated_postids:,}")

print(f"Full duplicate percentage: {full_duplicate_pct:.2f}%")
print(f"Repeated post ID percentage: {repeated_postid_pct:.2f}%")

print("Duplicate analysis completed successfully.")


=== DUPLICATE ANALYSIS ===
Total rows: 8,714,356
Fully duplicated rows: 1,579
Repeated post ID occurrences: 22,909
Full duplicate percentage: 0.02%
Repeated post ID percentage: 0.26%
Duplicate analysis completed successfully.



### 2.3 Investigation of Repeated Post IDs

**Objective:** Investigate repeated post identifiers and remove fully duplicated records.

From the complete dataset of 8,714,356 records, we identified:

- **1,579 fully duplicated records** (0.02%).
- **22,909 repeated post ID occurrences** (0.26%).

**Methodology:**

1. Create a working copy of the dataset.
2. Remove records that are identical across all attributes.
3. Investigate remaining records sharing the same `postid`.
4. Check whether repeated identifiers represent valid observations or conflicting information.

**Cleaning decision:**

Fully duplicated records are removed because they do not provide additional information.

However, records sharing the same `postid` are not automatically deleted, since they may represent observations from different countries or contain different attribute values.


In [5]:

# 2.3 Remove Fully Duplicated Records

# Create a cleaned working copy
df_clean = df.drop_duplicates().copy()

# Calculate cleaning statistics
original_rows = len(df)
cleaned_rows = len(df_clean)
removed_duplicates = original_rows - cleaned_rows

# Display results
print("=== DUPLICATE REMOVAL ===")
print(f"Original rows: {original_rows:,}")
print(f"Cleaned rows: {cleaned_rows:,}")
print(f"Removed duplicates: {removed_duplicates:,}")

# Verify that no fully duplicated records remain
assert df_clean.duplicated().sum() == 0

print("Duplicate removal completed successfully.")


=== DUPLICATE REMOVAL ===
Original rows: 8,714,356
Cleaned rows: 8,712,777
Removed duplicates: 1,579
Duplicate removal completed successfully.



### 2.4 Analysis of Conflicting Duplicate Records

**Objective:** Identify differences between records that share the same post identifier after removing fully duplicated rows.

**Methodology:**

1. Identify repeated `postid` values in the cleaned dataset.
2. Group records by `postid`.
3. Compare the number of distinct values in each attribute.
4. Count how many repeated-post groups contain conflicting values.

**Purpose:**

This analysis helps distinguish repeated observations from potentially inconsistent records.

Since identifiers may be anonymized differently across countries, differences will be investigated further using `(ioCountry, postid)` before making cleaning decisions.

**Outcome:** A summary of the attributes most frequently associated with repeated post IDs.


In [6]:

# 2.4 Analysis of Conflicting Duplicate Records

print("=== CONFLICTING DUPLICATE ANALYSIS ===")

# Step 1: Identify repeated post IDs after cleaning
duplicate_posts = df_clean[
    df_clean.duplicated(
        subset=["postid"],
        keep=False
    )
].copy()

print(f"Records with repeated post IDs: {len(duplicate_posts):,}")

print(
    f"Unique repeated post IDs: "
    f"{duplicate_posts['postid'].nunique():,}"
)

# Step 2: Compare attributes within repeated post IDs
conflicting_columns = (
    duplicate_posts
    .groupby("postid")
    .nunique(dropna=False)
    .gt(1)
    .sum()
    .sort_values(ascending=False)
)

# Step 3: Display results
conflict_summary = conflicting_columns.to_frame(
    name="Groups with different values"
)

display(conflict_summary)

print("Conflict analysis completed successfully.")


=== CONFLICTING DUPLICATE ANALYSIS ===
Records with repeated post IDs: 42,461
Unique repeated post IDs: 21,131


,Groups with different values
ioCountry,20838
follower_count,11228
following_count,9929
account_profile_description,409
accountid,234
account_creation_date,194
post_text,148
urls,117
hashtags,75
reposted_postid,67


Conflict analysis completed successfully.



### 2.5 Country-Level Post Consistency

**Objective:** Identify potential inconsistencies among records sharing the same country and post identifier.

**Methodology:**

1. Group records using `(ioCountry, postid)`.
2. Identify groups containing multiple account IDs.
3. Check inconsistencies in post text and timestamps.
4. Display conflicting records for further investigation.

**Purpose:**

Cross-country repetitions may be valid observations. Therefore, we focus on conflicts within the same country and post identifier to identify potentially inconsistent records.

**Outcome:** A summary of country-post groups requiring further validation.

No records are removed automatically at this stage.


In [7]:

# 2.5 Country-Level Post Consistency

print("=== COUNTRY-LEVEL POST CONSISTENCY ===")

# Group records by country and post ID
grouped = df_clean.groupby(
    ["ioCountry", "postid"],
    dropna=False
)

# Check account ID consistency
account_conflicts = grouped["accountid"].nunique(
    dropna=False
)

# Check post text consistency
text_conflicts = grouped["post_text"].nunique(
    dropna=False
)

# Check post time consistency
time_conflicts = grouped["post_time"].nunique(
    dropna=False
)

# Identify conflicting groups
conflict_summary = pd.DataFrame({
    "accountid_conflict": account_conflicts > 1,
    "post_text_conflict": text_conflicts > 1,
    "post_time_conflict": time_conflicts > 1
})

# Keep groups with at least one conflict
conflicting_groups = conflict_summary[
    conflict_summary.any(axis=1)
]

print(f"Groups with account ID conflicts: {(account_conflicts > 1).sum():,}")
print(f"Groups with post text conflicts: {(text_conflicts > 1).sum():,}")
print(f"Groups with timestamp conflicts: {(time_conflicts > 1).sum():,}")
print(f"Total groups with at least one conflict: {len(conflicting_groups):,}")

# Display examples
display(conflicting_groups.head(10))

print("Consistency analysis completed successfully.")


=== COUNTRY-LEVEL POST CONSISTENCY ===
Groups with account ID conflicts: 226
Groups with post text conflicts: 147
Groups with timestamp conflicts: 1
Total groups with at least one conflict: 296


accountid_conflict  \
ioCountry postid                                                           
ecuador   06590e1cdadd035275327726b0c203799f10da590a               False   
          0d0e7f0542efc16cbc65c2f61d6808611d65406b1d                True   
          17f61e99a642c36c4d46810638c7545e54816e5c9f                True   
          1c6cc92180b8586a93efd3cf5280ba1cd374d78426                True   
          1d502c947599040c31188a6d69a04350b97acf8fbe               False   
          1d6e232d8247d28bd95b1fc6df75cab64ba3cf2afa               False   
          215d80847cadb68dc5b27ec6d73fcabbf305c7622d               False   
          264bb4b4042cd5f46e5b9826b9d91f2ff4cba80bfa                True   
          28b6cba67fa61104d09589ae6f0f15596063aaea76                True   
          2b3c0f9ac6ba7304e575b6786e7f9689d1c04e452c                True   

                                                      post_text_conflict  \
ioCountry postid                                                           
ecuador   06590e1cdadd035275327726b0c203799f10da590a                True   
          0d0e7f0542efc16cbc65c2f61d6808611d65406b1d               False   
          17f61e99a642c36c4d46810638c7545e54816e5c9f                True   
          1c6cc92180b8586a93efd3cf5280ba1cd374d78426               False   
          1d502c947599040c31188a6d69a04350b97acf8fbe                True   
          1d6e232d8247d28bd95b1fc6df75cab64ba3cf2afa                True   
          215d80847cadb68dc5b27ec6d73fcabbf305c7622d                True   
          264bb4b4042cd5f46e5b9826b9d91f2ff4cba80bfa                True   
          28b6cba67fa61104d09589ae6f0f15596063aaea76               False   
          2b3c0f9ac6ba7304e575b6786e7f9689d1c04e452c               False   

                                                      post_time_conflict  
ioCountry postid                                                          
ecuador   06590e1cdadd035275327726b0c203799f10da590a               False  
          0d0e7f0542efc16cbc65c2f61d6808611d65406b1d               False  
          17f61e99a642c36c4d46810638c7545e54816e5c9f               False  
          1c6cc92180b8586a93efd3cf5280ba1cd374d78426               False  
          1d502c947599040c31188a6d69a04350b97acf8fbe               False  
          1d6e232d8247d28bd95b1fc6df75cab64ba3cf2afa               False  
          215d80847cadb68dc5b27ec6d73fcabbf305c7622d               False  
          264bb4b4042cd5f46e5b9826b9d91f2ff4cba80bfa               False  
          28b6cba67fa61104d09589ae6f0f15596063aaea76               False  
          2b3c0f9ac6ba7304e575b6786e7f9689d1c04e452c               False

Consistency analysis completed successfully.



### 2.6 Flagging Conflicting Records

**Objective:** Identify records associated with inconsistent country-post combinations.

**Methodology:**

1. Use the conflicting `(ioCountry, postid)` groups identified in Section 2.5.
2. Retrieve all records belonging to these groups.
3. Classify each group according to its conflict type.
4. Save a separate report for further investigation.

**Cleaning Decision:**

Conflicting records are retained in the cleaned dataset because the available information is insufficient to determine which observation is correct.

Rather than removing potentially valid data, these records are flagged for further validation.

**Outcome:** A separate conflict report containing affected records and their corresponding inconsistency types.


In [8]:

# 2.6 Flagging Conflicting Records

from pathlib import Path

print("=== CONFLICTING RECORDS REPORT ===")

# Step 1: Retrieve records belonging to conflicting groups
conflict_keys = conflicting_groups.reset_index()

conflict_records = df_clean.merge(
    conflict_keys,
    on=["ioCountry", "postid"],
    how="inner",
    validate="many_to_one"
)

# Step 2: Count affected records
print(f"Conflicting groups: {len(conflicting_groups):,}")
print(f"Affected records: {len(conflict_records):,}")

# Step 3: Verify conflict classification
print("\nConflict types:")

for column in [
    "accountid_conflict",
    "post_text_conflict",
    "post_time_conflict"
]:
    print(f"{column}: {conflict_records[column].sum():,} records")

# Step 4: Save conflict report
output_dir = Path("../data/data_processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "conflicting_records.csv"

conflict_records.to_csv(
    output_path,
    index=False
)

print(f"\nReport saved to: {output_path}")

# Step 5: Display examples
display(
    conflict_records[
        [
            "ioCountry",
            "postid",
            "accountid",
            "accountid_conflict",
            "post_text_conflict",
            "post_time_conflict"
        ]
    ].head(10)
)

print("\nConflict report generated successfully.")


=== CONFLICTING RECORDS REPORT ===
Conflicting groups: 296
Affected records: 601

Conflict types:
accountid_conflict: 460 records
post_text_conflict: 295 records
post_time_conflict: 2 records

Report saved to: ../data/data_processed/conflicting_records.csv


,ioCountry,postid,accountid,accountid_conflict,post_text_conflict,post_time_conflict
0,ecuador,c62bfbf48df48628bf1764a380691ca74ac6a858fe,b11edd9b878259a855e0cecfcb532cef711937bfda,True,False,False
1,ecuador,c62bfbf48df48628bf1764a380691ca74ac6a858fe,9743f71178cf2ed5e9ff9100700f6f6307839a2560,True,False,False
2,ecuador,4262f7363a33ef07b7f47e26da8a7d940cb5d10c25,1a662c6d89d6aa1ddd44c94bbce18f7a2c84d344b1,True,False,False
3,ecuador,4262f7363a33ef07b7f47e26da8a7d940cb5d10c25,eaeeba8dacb04092abdf1854205a56ec03cfea5c05,True,False,False
4,ecuador,f6bb4c21abe4052d8fa4d606efa64afd39b79c5eba,b11edd9b878259a855e0cecfcb532cef711937bfda,True,False,False
5,ecuador,f6bb4c21abe4052d8fa4d606efa64afd39b79c5eba,6dc84f358e14d31f8a3ce339b246ce2eea2432b1b5,True,False,False
6,ecuador,df2346ef01a2579f80d9c70d8553ad0f1ad7fcafe1,48126ba5d1e3ec30b69201b9cda6a58be7b71a32b2,True,False,False
7,ecuador,df2346ef01a2579f80d9c70d8553ad0f1ad7fcafe1,9743f71178cf2ed5e9ff9100700f6f6307839a2560,True,False,False
8,ecuador,330a30ec691b83588ce22264dcfb26a6a99c232f91,b11edd9b878259a855e0cecfcb532cef711937bfda,True,False,False
9,ecuador,330a30ec691b83588ce22264dcfb26a6a99c232f91,f6f6a4a0469d64ff2bdb1ed81dac77554226dd8c29,True,False,False



Conflict report generated successfully.



### 2.7 Data-Type Validation

**Objective:** Verify that each attribute uses an appropriate data type.

**Methodology:**

1. Inspect the current data type of each attribute.
2. Identify numeric attributes that may have been imported as strings.
3. Check whether date attributes can be converted to datetime.
4. Identify columns requiring further type conversion.

**Purpose:**

Incorrect data types may affect statistical analysis, comparisons, filtering, and data cleaning.

This step identifies potential type issues without modifying the original values.

**Outcome:** A summary of dataset attributes and their current data types.


In [9]:

# 2.7 Data-Type Validation

print("=== DATA-TYPE VALIDATION ===")

# Inspect data types in the cleaned dataset
dtype_summary = pd.DataFrame({
    "Data Type": df_clean.dtypes.astype(str),
    "Non-Null Count": df_clean.notna().sum(),
    "Missing Count": df_clean.isna().sum()
})

display(dtype_summary)

# Identify columns stored as object
object_columns = df_clean.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nColumns stored as object:")
print(object_columns)

print("\nData-type inspection completed successfully.")


=== DATA-TYPE VALIDATION ===


,Data Type,Non-Null Count,Missing Count
ioCountry,str,8712777,0
postid,str,8712777,0
post_text,str,8712774,3
application_name,str,8712777,0
post_language,str,8609050,103727
in_reply_to_postid,str,886078,7826699
in_reply_to_accountid,str,874926,7837851
post_time,str,8712777,0
accountid,str,8712777,0
account_profile_description,str,7043165,1669612



Columns stored as object:
['ioCountry', 'postid', 'post_text', 'application_name', 'post_language', 'in_reply_to_postid', 'in_reply_to_accountid', 'post_time', 'accountid', 'account_profile_description', 'follower_count', 'following_count', 'account_creation_date', 'reposted_accountid', 'reposted_postid', 'hashtags', 'urls', 'account_mentions', 'is_control']

Data-type inspection completed successfully.


/var/folders/q9/zj0sb7h14b1513lt554dnmy80000gn/T/ipykernel_9054/1234938142.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df_clean.select_dtypes(



### 2.8 Data-Type Conversion and Validation

**Objective:** Validate and convert attributes to appropriate data types for further analysis.

**Methodology:**

1. Convert follower and following counts into numeric values.
2. Convert post and account creation dates into datetime values.
3. Identify values that cannot be converted successfully.
4. Inspect the distinct values of `is_control` before Boolean conversion.

**Purpose:**

Correct data types are necessary for reliable statistical calculations, chronological comparisons, and data consistency checks.

Invalid values are identified before applying any cleaning decisions.

**Outcome:** Validated numeric and datetime attributes, together with a summary of conversion issues.


In [10]:

# 2.8 Data-Type Conversion and Validation


print("=== DATA-TYPE CONVERSION ===")

# Step 1: Validate numeric columns
numeric_columns = ["follower_count", "following_count"]

for col in numeric_columns:
    original = df_clean[col]
    converted = pd.to_numeric(original, errors="coerce")

    invalid_count = (
        original.notna() & converted.isna()
    ).sum()

    # Check fractional values before using nullable integers
    fractional_count = (
        converted.notna() &
        (converted % 1 != 0)
    ).sum()

    print(f"\n{col}")
    print(f"Invalid numeric values: {invalid_count:,}")
    print(f"Fractional values: {fractional_count:,}")

    # Preserve converted values without forcing integer conversion
    df_clean[col] = converted

# Step 2: Validate datetime columns
date_columns = [
    "post_time",
    "account_creation_date"
]

for col in date_columns:
    original = df_clean[col]

    converted = pd.to_datetime(
        original,
        errors="coerce",
        utc=True,
        format="mixed"
    )

    invalid_count = (
        original.notna() & converted.isna()
    ).sum()

    print(f"\n{col}")
    print(f"Invalid datetime values: {invalid_count:,}")

    df_clean[col] = converted

# Step 3: Inspect Boolean values
print("\n=== IS_CONTROL VALUES ===")

display(
    df_clean["is_control"]
    .value_counts(dropna=False)
)

# Step 4: Verify resulting data types
print("\n=== UPDATED DATA TYPES ===")

display(
    df_clean[
        numeric_columns + date_columns + ["is_control", "is_repost"]
    ].dtypes
)

print("\nData-type validation completed.")


=== DATA-TYPE CONVERSION ===

follower_count
Invalid numeric values: 2,265
Fractional values: 0

following_count
Invalid numeric values: 2,160
Fractional values: 0

post_time
Invalid datetime values: 0

account_creation_date
Invalid datetime values: 0

=== IS_CONTROL VALUES ===


is_control
False          5229135
True           3482530
is_control?       1112
Name: count, dtype: int64


=== UPDATED DATA TYPES ===


follower_count                       float64
following_count                      float64
post_time                datetime64[us, UTC]
account_creation_date    datetime64[us, UTC]
is_control                               str
is_repost                               bool
dtype: object


Data-type validation completed.



### 2.9 Investigation of Invalid Attribute Values

**Objective:** Investigate values that do not match the expected numeric or Boolean formats.

**Methodology:**

1. Identify invalid values in follower and following counts.
2. Inspect their original representations in the raw dataset.
3. Examine unexpected values in the `is_control` attribute.
4. Determine whether invalid values can be corrected or should be treated as missing.

**Purpose:**

Unexpected values may originate from data collection, formatting inconsistencies, or missing-value placeholders.

Inspecting these values before cleaning helps avoid introducing incorrect information into the dataset.

**Outcome:** A summary of invalid numeric and Boolean values requiring treatment.


In [11]:

# 2.9 Investigation of Invalid Attribute Values


print("=== INVALID VALUE INVESTIGATION ===")

# Investigate original numeric values
for col in ["follower_count", "following_count"]:

    original_values = df.loc[df_clean.index, col]
    converted_values = pd.to_numeric(
        original_values,
        errors="coerce"
    )

    invalid_mask = (
        original_values.notna() &
        converted_values.isna()
    )

    print(f"\n{col}")
    print(f"Invalid values: {invalid_mask.sum():,}")

    print("Most frequent invalid representations:")
    display(
        original_values[invalid_mask]
        .value_counts(dropna=False)
        .head(15)
    )

# Investigate is_control
print("\n=== IS_CONTROL DISTRIBUTION ===")

display(
    df_clean["is_control"]
    .value_counts(dropna=False)
)

print("\nInvalid-value investigation completed.")


=== INVALID VALUE INVESTIGATION ===

follower_count
Invalid values: 2,265
Most frequent invalid representations:


follower_count
error      281
unknown    268
NaN-ish    258
none       256
many       236
4O00       210
12abc      202
1,2,3      193
??42       191
ten        170
Name: count, dtype: int64


following_count
Invalid values: 2,160
Most frequent invalid representations:


following_count
NaN-ish    258
unknown    249
many       244
none       243
error      227
12abc      212
ten        200
??42       193
1,2,3      167
4O00       167
Name: count, dtype: int64


=== IS_CONTROL DISTRIBUTION ===


is_control
False          5229135
True           3482530
is_control?       1112
Name: count, dtype: int64


Invalid-value investigation completed.



### 2.10 Recovery of Invalid Boolean Values

**Objective:** Recover invalid `is_control` values using reliable information from other records belonging to the same account within the same country.

**Methodology:**

1. Convert valid `True` and `False` values into Boolean format.
2. Treat invalid values such as `is_control?` as missing.
3. Group records by `(ioCountry, accountid)`.
4. Identify accounts with exactly one consistent known classification.
5. Recover missing labels only for accounts with a unique valid classification.
6. Preserve unresolved values as missing.

**Cleaning Decision:**

Account identifiers may be anonymized differently across countries. Therefore, recovery is restricted to records sharing both the same country and account identifier.

Accounts containing conflicting known classifications are excluded from recovery.

**Outcome:** A nullable Boolean column containing validated and recovered `is_control` values.


In [12]:

# 2.10 Recover Invalid is_control Values
# Country-aware account recovery


print("=== COUNTRY-AWARE IS_CONTROL RECOVERY ===")

# Step 1: Restore the original values for the cleaned rows
original_control = (
    df.loc[df_clean.index, "is_control"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Step 2: Convert valid labels into Boolean values
control_mapping = {
    "true": True,
    "false": False
}

df_clean["is_control"] = (
    original_control
    .map(control_mapping)
    .astype("boolean")
)

missing_before = df_clean["is_control"].isna().sum()

print(f"Missing values before recovery: {missing_before:,}")

# Step 3: Find accounts with consistent known labels
account_labels = (
    df_clean.dropna(subset=["is_control"])
    .groupby(
        ["ioCountry", "accountid"],
        dropna=False
    )["is_control"]
    .agg(["nunique", "first"])
)

# Keep accounts with exactly one known label
consistent_accounts = account_labels.loc[
    account_labels["nunique"] == 1, "first"
]

# Step 4: Recover missing labels using country + account ID
missing_mask = df_clean["is_control"].isna()

missing_keys = pd.MultiIndex.from_frame(
    df_clean.loc[
        missing_mask,
        ["ioCountry", "accountid"]
    ]
)

recovered_values = pd.array(
    consistent_accounts.reindex(missing_keys),
    dtype="boolean"
)

df_clean.loc[missing_mask, "is_control"] = recovered_values

# Step 5: Verify recovery
missing_after = df_clean["is_control"].isna().sum()
recovered_count = missing_before - missing_after

print(f"Recovered values: {recovered_count:,}")
print(f"Remaining missing values: {missing_after:,}")

print("\nFinal distribution:")
display(df_clean["is_control"].value_counts(dropna=False))

# Step 6: Confirm data type
print("\nData type:", df_clean["is_control"].dtype)

assert str(df_clean["is_control"].dtype) == "boolean"

print("\nCountry-aware Boolean recovery completed.")


=== COUNTRY-AWARE IS_CONTROL RECOVERY ===
Missing values before recovery: 1,112
Recovered values: 1,110
Remaining missing values: 2

Final distribution:


is_control
False    5229829
True     3482946
<NA>           2
Name: count, dtype: Int64


Data type: boolean

Country-aware Boolean recovery completed.



### 2.11 Recovery of Missing Numeric Values

**Objective:** Recover missing follower and following counts using valid observations from the same account.

**Methodology:**

1. Identify missing numeric values in `follower_count` and `following_count`.
2. Match records using `(ioCountry, accountid)`.
3. Use `post_time` to identify the closest valid observation for each missing value.
4. Recover values only when a valid observation exists within the same country and account.
5. Preserve unresolved values as missing.

**Purpose:**

Follower and following counts can vary over time. Therefore, using the nearest available observation is more appropriate than replacing missing values with a global mean or median.

**Outcome:** A dataset with fewer missing numeric values while preserving account-level and temporal consistency.


In [13]:

# 2.11 Recover Missing Numeric Values

print("=== NUMERIC MISSING-VALUE RECOVERY ===")

numeric_columns = ["follower_count", "following_count"]
group_columns = ["ioCountry", "accountid"]

for col in numeric_columns:

    missing_before = df_clean[col].isna().sum()

    print(f"\nProcessing {col}")
    print(f"Missing before recovery: {missing_before:,}")

    
    missing_rows = df_clean.loc[
        df_clean[col].isna(),
        group_columns + ["post_time"]
    ].copy()

    
    valid_rows = df_clean.loc[
        df_clean[col].notna(),
        group_columns + ["post_time", col]
    ].copy()

    
    missing_rows["_row_id"] = missing_rows.index

    
    missing_rows = missing_rows.sort_values("post_time")
    valid_rows = valid_rows.sort_values("post_time")

    
    recovered = pd.merge_asof(
        missing_rows,
        valid_rows,
        on="post_time",
        by=group_columns,
        direction="nearest",
        allow_exact_matches=True
    )

    
    df_clean.loc[
        recovered["_row_id"],
        col
    ] = recovered[col].to_numpy()

    missing_after = df_clean[col].isna().sum()

    print(f"Recovered values: {missing_before - missing_after:,}")
    print(f"Remaining missing: {missing_after:,}")

print("\nNumeric recovery completed.")


=== NUMERIC MISSING-VALUE RECOVERY ===

Processing follower_count
Missing before recovery: 3,583
Recovered values: 3,581
Remaining missing: 2

Processing following_count
Missing before recovery: 3,352
Recovered values: 3,349
Remaining missing: 3

Numeric recovery completed.



### 2.12 Validation of Recovered Numeric Values

**Objective:** Verify the consistency of follower and following counts after missing-value recovery.

**Methodology:**

1. Count remaining missing values.
2. Identify negative values, which are invalid for account counts.
3. Identify fractional values, since follower and following counts should be integers.
4. Examine the minimum, maximum, and selected percentiles to identify potentially anomalous values.

**Purpose:**

Successfully recovering missing values does not guarantee that all numeric observations are valid. Additional checks help detect errors before subsequent analysis.

**Outcome:** A validation summary of both numeric attributes without automatically deleting extreme observations.


In [14]:

# 2.12 Validation of Recovered Numeric Values

print("=== NUMERIC VALUE VALIDATION ===")

numeric_columns = ["follower_count", "following_count"]

results = []

for col in numeric_columns:
    values = df_clean[col]

    results.append({
        "Attribute": col,
        "Missing": values.isna().sum(),
        "Negative": (values < 0).sum(),
        "Fractional": (
            values.notna() & (values % 1 != 0)
        ).sum(),
        "Minimum": values.min(),
        "Median": values.median(),
        "99th Percentile": values.quantile(0.99),
        "Maximum": values.max()
    })

validation_summary = pd.DataFrame(results)

display(validation_summary)

print("\nNumeric validation completed.")


=== NUMERIC VALUE VALIDATION ===


,Attribute,Missing,Negative,Fractional,Minimum,Median,99th Percentile,Maximum
0,follower_count,2,977,0,-995298.0,714.0,134805.0,9.958279e+10
1,following_count,3,1029,0,-999963.0,499.0,29070.0,9.991536e+10



Numeric validation completed.



### 2.13 Investigation of Extreme Numeric Values

**Objective:** Identify potentially unrealistic values in follower and following counts.

**Methodology:**

1. Examine the distribution of numeric attributes.
2. Count observations exceeding selected diagnostic thresholds.
3. Compare extreme values with the 99th percentile.
4. Inspect records containing extremely large values.

**Purpose:**

Extremely large values may indicate data corruption or collection errors. However, high values are not automatically considered invalid because social-media account popularity can vary significantly.

**Cleaning Decision:**

No values are removed automatically. Extreme observations are investigated before selecting an appropriate treatment.

**Outcome:** A summary of potentially anomalous numeric values requiring further validation.


In [15]:

# 2.13 Investigation of Extreme Numeric Values


print("=== EXTREME NUMERIC VALUE ANALYSIS ===")

numeric_columns = ["follower_count", "following_count"]

# Diagnostic thresholds, not automatic deletion limits
thresholds = [1_000_000, 10_000_000, 100_000_000, 1_000_000_000]

results = []

for col in numeric_columns:
    values = df_clean[col]

    for threshold in thresholds:
        results.append({
            "Attribute": col,
            "Threshold": threshold,
            "Records Above Threshold": int(
                (values > threshold).sum()
            ),
            "Percentage (%)": round(
                (values > threshold).mean() * 100, 4
            )
        })

threshold_summary = pd.DataFrame(results)

display(threshold_summary)

# Inspect the largest observations
for col in numeric_columns:
    print(f"\nTop 10 largest values in {col}:")

    display(
        df_clean[
            ["ioCountry", "accountid", "post_time", col]
        ].nlargest(10, col)
    )

print("\nExtreme-value investigation completed.")


=== EXTREME NUMERIC VALUE ANALYSIS ===


,Attribute,Threshold,Records Above Threshold,Percentage (%)
0,follower_count,1000000,16045,0.1842
1,follower_count,10000000,1312,0.0151
2,follower_count,100000000,493,0.0057
3,follower_count,1000000000,493,0.0057
4,following_count,1000000,561,0.0064
5,following_count,10000000,518,0.0059
6,following_count,100000000,518,0.0059
7,following_count,1000000000,518,0.0059



Top 10 largest values in follower_count:


,ioCountry,accountid,post_time,follower_count
5043186,russia_1,4b262547190219226e276a7e6d342bc0cc42c9e81d,2012-07-04 16:00:06+00:00,9.958279e+10
564116,ecuador,695748fd7fe2d32aff1ee129219199ce412bde3626,2012-08-06 22:08:49+00:00,9.893545e+10
7324189,uae,6dd25ded1bd6188f6fca6beb438fbfa8bb1e3c12cd,2018-09-24 11:22:00+00:00,9.871636e+10
2510911,iran_1,ce3438d4a69ae98420a7a3c449366da079d00df09b,2014-09-09 19:04:00+00:00,9.850715e+10
7115197,uae,4be31b86260c6fdaf06554e5f808c2e0ad3ffff714,2018-06-14 16:06:00+00:00,9.832753e+10
2456369,iran_1,ce3438d4a69ae98420a7a3c449366da079d00df09b,2014-07-16 07:21:00+00:00,9.820524e+10
1025307,ecuador,a35a511e26bc2643145165bab63a5a192228f90138,2019-03-03 03:28:05+00:00,9.793376e+10
322427,ecuador,2ef928f0ed37baa489e9dffd62f94838e174a015f3,2018-04-13 12:00:00+00:00,9.791654e+10
7249933,uae,b9e82f4cfbea7c13563f35a06b50952a20ab4a8bf5,2018-08-18 23:42:26+00:00,9.788533e+10
4028295,russia_1,7e363e1de5b0ed81518fb5f81e5f9eceb30c353b79,2013-10-10 14:12:25+00:00,9.784619e+10



Top 10 largest values in following_count:


,ioCountry,accountid,post_time,following_count
2830385,iran_4,9f19a9ce73914b9bd92ecbae3f37e0bb0025e5ac00,2014-11-16 23:55:49+00:00,9.991536e+10
7225632,uae,65ea9489070a939223eb5711a24e28e69cc44dbe54,2018-08-05 11:24:00+00:00,9.986612e+10
1153746,ecuador,06ae71cd26b039f685ffa49c64780bd86f18fdff94,2019-04-08 15:50:00+00:00,9.979599e+10
3266683,iran_4,85c91bfb9fe420a08d0df5e97465067d39fb72fe3f,2018-03-07 14:28:00+00:00,9.975606e+10
4987812,russia_1,212f034522aa94b0065565821d583b12380cccdc96,2014-08-06 10:54:14+00:00,9.964013e+10
3654588,russia_1,4a177c0b39e845b05b8b8196204aa9844ed0a578ff,2013-02-13 21:31:34+00:00,9.945840e+10
34278,ecuador,40a927417ec7dd26939b28d3e7eeb83eddd908f1c3,2011-11-23 01:48:00+00:00,9.942571e+10
5107577,russia_1,c52d67acaa4c0b8c8b84de7d3b08fb0a6ba3d2cf69,2014-08-15 12:12:00+00:00,9.889349e+10
1040523,ecuador,9c77c90d9cde0cb6998c94d004453e17da4228fe82,2019-03-10 23:58:31+00:00,9.887374e+10
8465520,uae,46b2ea37016db78768b66aace8037a931ad406413c,2017-10-15 10:05:00+00:00,9.874355e+10



Extreme-value investigation completed.



### 2.14 Validation of Suspicious Extreme Values

**Objective:** Investigate potentially corrupted follower and following counts exceeding one billion.

**Methodology:**

1. Identify observations with numeric values above one billion.
2. Compare suspicious values against the original dataset.
3. Distinguish original extreme values from values introduced during missing-value recovery.
4. Count affected accounts using the combination of `ioCountry` and `accountid`.

**Purpose:**

Extremely large follower and following counts may represent data collection errors or corrupted values.

Before cleaning, it is necessary to determine whether these values were already present in the raw dataset or propagated through the recovery process.

**Cleaning Decision:**

Values are investigated and flagged rather than immediately removed.

**Outcome:** A report describing the origin and distribution of suspicious numeric values.


In [16]:

# 2.14 Validation of Suspicious Extreme Values


print("=== SUSPICIOUS NUMERIC VALUES ===")

THRESHOLD = 1_000_000_000

numeric_columns = [
    "follower_count",
    "following_count"
]

for col in numeric_columns:

    # Identify suspicious values in cleaned dataset
    suspicious_mask = df_clean[col] > THRESHOLD

    # Retrieve corresponding original values
    original_values = pd.to_numeric(
        df.loc[df_clean.index, col],
        errors="coerce"
    )

    # Identify whether suspicious values existed originally
    original_extreme = original_values > THRESHOLD

    # Separate original and potentially propagated values
    original_count = (
        suspicious_mask & original_extreme
    ).sum()

    introduced_count = (
        suspicious_mask & ~original_extreme
    ).sum()

    # Count affected accounts
    affected_accounts = (
        df_clean.loc[
            suspicious_mask,
            ["ioCountry", "accountid"]
        ]
        .drop_duplicates()
        .shape[0]
    )

    print(f"\nAttribute: {col}")
    print(f"Suspicious values: {suspicious_mask.sum():,}")
    print(f"Already extreme in original data: {original_count:,}")
    print(f"Not extreme in original data: {introduced_count:,}")
    print(f"Affected country-account combinations: {affected_accounts:,}")

print("\nExtreme-value validation completed.")


=== SUSPICIOUS NUMERIC VALUES ===

Attribute: follower_count
Suspicious values: 493
Already extreme in original data: 493
Not extreme in original data: 0
Affected country-account combinations: 431

Attribute: following_count
Suspicious values: 518
Already extreme in original data: 518
Not extreme in original data: 0
Affected country-account combinations: 455

Extreme-value validation completed.



### 2.15 Treatment of Suspicious Numeric Values

**Objective:** Handle implausibly large values in follower and following counts without removing valid records.

**Methodology:**

1. Identify follower and following counts exceeding the selected diagnostic threshold of one billion.
2. Record the affected observations for auditing.
3. Replace suspicious numeric values with missing values (`NaN`).
4. Preserve all other attributes and observations.
5. Recalculate missing-value counts after treatment.

**Cleaning Decision:**

Values exceeding one billion are treated as suspected data corruption due to their implausible magnitude relative to the dataset.

The threshold is a conservative, dataset-specific cleaning assumption rather than a universal statistical rule.

**Outcome:** Suspicious values are excluded from subsequent statistical analysis and can be considered for recovery using validated observations.


In [17]:

# 2.15 Treatment of Suspicious Numeric Values


print("=== SUSPICIOUS NUMERIC VALUE TREATMENT ===")

THRESHOLD = 1_000_000_000

numeric_columns = [
    "follower_count",
    "following_count"
]

# Step 1: Preserve suspicious values in an audit report
suspicious_mask = (
    df_clean[numeric_columns] > THRESHOLD
).any(axis=1)

suspicious_records = df_clean.loc[
    suspicious_mask,
    ["ioCountry", "postid", "accountid",
     "post_time"] + numeric_columns
].copy()

print(f"Records affected: {len(suspicious_records):,}")

# Step 2: Replace suspicious numeric values with NaN
for col in numeric_columns:

    invalid_mask = df_clean[col] > THRESHOLD

    invalid_count = int(invalid_mask.sum())

    print(f"\n{col}")
    print(f"Suspicious values found: {invalid_count:,}")

    df_clean.loc[invalid_mask, col] = float("nan")

    print(
        f"Remaining extreme values: "
        f"{(df_clean[col] > THRESHOLD).sum():,}"
    )

# Step 3: Verify treatment
print("\n=== MISSING VALUES AFTER TREATMENT ===")

display(df_clean[numeric_columns].isna().sum())

print("\nSuspicious-value treatment completed.")


=== SUSPICIOUS NUMERIC VALUE TREATMENT ===
Records affected: 1,011

follower_count
Suspicious values found: 493
Remaining extreme values: 0

following_count
Suspicious values found: 518
Remaining extreme values: 0

=== MISSING VALUES AFTER TREATMENT ===


follower_count     495
following_count    521
dtype: int64


Suspicious-value treatment completed.


In [18]:

# Verify original extreme values and current cleaning state

numeric_columns = ["follower_count", "following_count"]
THRESHOLD = 1_000_000_000

for col in numeric_columns:
    original = pd.to_numeric(
        df.loc[df_clean.index, col],
        errors="coerce"
    )

    print(f"\n{col}")
    print(f"Original extreme values: {(original > THRESHOLD).sum():,}")
    print(f"Current extreme values: {(df_clean[col] > THRESHOLD).sum():,}")
    print(f"Current missing values: {df_clean[col].isna().sum():,}")



follower_count
Original extreme values: 493
Current extreme values: 0
Current missing values: 495

following_count
Original extreme values: 518
Current extreme values: 0
Current missing values: 521



### 2.16 Recovery After Extreme-Value Treatment

**Objective:** Recover missing follower and following counts after excluding implausibly large values.

**Methodology:**

1. Identify missing numeric values remaining after outlier treatment.
2. Match observations using `(ioCountry, accountid)`.
3. Select the nearest valid observation based on `post_time`.
4. Recover values only from observations containing valid numeric counts.
5. Preserve values that cannot be recovered as missing.

**Purpose:**

The recovery process uses temporal and account-level information while preventing previously identified extreme values from propagating into the cleaned dataset.

**Outcome:** A reduction in missing numeric values without introducing the excluded extreme observations.


In [19]:

# 2.16 Recover Missing Numeric Values After Outlier Treatment


print("=== POST-OUTLIER NUMERIC RECOVERY ===")

numeric_columns = ["follower_count", "following_count"]
group_columns = ["ioCountry", "accountid"]
THRESHOLD = 1_000_000_000

for col in numeric_columns:

    missing_before = int(df_clean[col].isna().sum())

    # Missing observations
    missing_rows = df_clean.loc[
        df_clean[col].isna(),
        group_columns + ["post_time"]
    ].copy()

    missing_rows["_row_id"] = missing_rows.index

    # Valid reference observations only
    valid_mask = (
        df_clean[col].notna() &
        df_clean[col].between(0, THRESHOLD)
    )

    valid_rows = df_clean.loc[
        valid_mask,
        group_columns + ["post_time", col]
    ].copy()

    # Sort by timestamp for merge_asof
    missing_rows = missing_rows.sort_values("post_time")
    valid_rows = valid_rows.sort_values("post_time")

    # Match nearest valid observation
    recovered = pd.merge_asof(
        missing_rows,
        valid_rows,
        on="post_time",
        by=group_columns,
        direction="nearest"
    )

    # Fill only missing values
    df_clean.loc[
        recovered["_row_id"],
        col
    ] = recovered[col].to_numpy()

    missing_after = int(df_clean[col].isna().sum())

    print(f"\n{col}")
    print(f"Missing before: {missing_before:,}")
    print(f"Recovered: {missing_before - missing_after:,}")
    print(f"Remaining missing: {missing_after:,}")

    # Ensure suspicious values were not reintroduced
    assert not (df_clean[col] > THRESHOLD).any()

print("\nPost-outlier recovery completed successfully.")


=== POST-OUTLIER NUMERIC RECOVERY ===

follower_count
Missing before: 495
Recovered: 493
Remaining missing: 2

following_count
Missing before: 521
Recovered: 518
Remaining missing: 3

Post-outlier recovery completed successfully.



## 3. Additional Duplicate Record Investigation

### 3.1 Important Clarifications About Duplicate Records

**Objective:** Understand the limitations of anonymized identifiers before deciding how to handle conflicting records.

**Important considerations:**

- **Anonymized identifiers:** User IDs, post IDs, and URLs have been anonymized using hash functions. The anonymization may differ between `ioCountry` groups, so identifiers are not necessarily comparable across countries.

- **Country-level conflicts:** Records sharing the same `(ioCountry, postid)` but containing different `accountid` values may indicate data inconsistencies.

- **Other conflicting attributes:** Differences in `post_text`, `post_time`, and `is_control` require further investigation before making cleaning decisions.

**Cleaning Decision:**

Records with conflicting information are retained unless sufficient evidence is available to identify an incorrect observation.



### 3.2 Investigation of Conflicting Account Histories

**Objective:** Examine the posting histories of accounts associated with conflicting post identifiers.

**Methodology:**

1. Select an example of a country-post combination with conflicting account IDs.
2. Retrieve the records associated with this combination.
3. Identify the accounts involved.
4. Examine other posts from these accounts within the same country.

**Purpose:**

Comparing account histories may provide additional context about conflicting identifiers.

However, differences in posting behavior alone cannot establish which account identifier is correct.

**Outcome:** A manual case study of conflicting records without modifying the cleaned dataset.


In [20]:

# 3.2 Investigation of Conflicting Account Histories

print("=== CONFLICTING ACCOUNT HISTORY INVESTIGATION ===")

# Step 1: Use the conflict groups identified in Section 2.5
account_conflict_groups = conflicting_groups[
    conflicting_groups["accountid_conflict"]
]

if account_conflict_groups.empty:
    print("No account ID conflicts found.")

else:
    # Step 2: Select the first conflicting group
    country, post_id = account_conflict_groups.index[0]

    # Step 3: Retrieve the conflicting records from df_clean
    pair = df_clean.loc[
        (df_clean["ioCountry"] == country) &
        (df_clean["postid"] == post_id)
    ]

    print(f"Country: {country}")
    print(f"Post ID: {post_id}")
    print(f"Conflicting accounts: {pair['accountid'].nunique()}")

    display(
        pair[["ioCountry", "postid", "accountid", "post_text"]]
    )

    # Step 4: Investigate other posts from these accounts
    accounts = pair["accountid"].dropna().unique()

    for account in accounts:

        account_posts = df_clean.loc[
            (df_clean["ioCountry"] == country) &
            (df_clean["accountid"] == account) &
            (df_clean["postid"] != post_id),
            ["post_time", "post_text", "post_language"]
        ]

        print(f"\nAccount: {account}")
        print(f"Other posts found: {len(account_posts):,}")

        display(account_posts.head(10))

print("\nAccount-history investigation completed.")


=== CONFLICTING ACCOUNT HISTORY INVESTIGATION ===
Country: ecuador
Post ID: 0d0e7f0542efc16cbc65c2f61d6808611d65406b1d
Conflicting accounts: 2


,ioCountry,postid,accountid,post_text
1423338,ecuador,0d0e7f0542efc16cbc65c2f61d6808611d65406b1d,fa3fe8929f0583d879d124ed805a103368afe1f43c,En menos de 2 meses Cedatos recibió $807.360 d...
1423343,ecuador,0d0e7f0542efc16cbc65c2f61d6808611d65406b1d,a5cad6152d2da18181b8eeccaf7c3c94c22616f3f1,En menos de 2 meses Cedatos recibió $807.360 d...



Account: fa3fe8929f0583d879d124ed805a103368afe1f43c
Other posts found: 84


,post_time,post_text,post_language
1422855,2017-03-21 14:52:41+00:00,Compañero @15f03bf2627f7dd4e6fa57f6676e28ffbda...,es
1422856,2017-03-21 14:52:48+00:00,Informamos desde @8a918317fa35e2ba741a9fb6bddd...,es
1422857,2017-03-21 14:52:52+00:00,Con los traficantes no habrá impunidad. Hemos ...,es
1422858,2017-03-21 14:52:56+00:00,.@2087399d0b911ed0bb43aa9da21d7301f0a077e56c: ...,es
1422859,2017-03-21 14:52:59+00:00,Se viene la #GranMingaAgropecuaria entregaremo...,es
1422860,2017-03-21 14:53:02+00:00,Saludos amigos-as! Les comparto el mensaje sem...,es
1422861,2017-03-21 14:53:04+00:00,.@aae2514dd2b49b2e9005f27a9b7c88b4368e7b0c2d: ...,es
1422862,2017-03-21 14:53:07+00:00,En instantes siga la entrevista en vivo a @b28...,es
1422863,2017-03-21 14:53:09+00:00,Te invitamos a escuchar a @c09b5b44e08e1dee035...,es
1422864,2017-03-21 14:53:20+00:00,Como legisladores debemos fortalecer la Consti...,es



Account: a5cad6152d2da18181b8eeccaf7c3c94c22616f3f1
Other posts found: 83


,post_time,post_text,post_language
1422876,2017-03-21 14:54:45+00:00,Informamos desde @8a918317fa35e2ba741a9fb6bddd...,es
1422877,2017-03-21 14:55:14+00:00,Con los traficantes no habrá impunidad. Hemos ...,es
1422878,2017-03-21 14:55:18+00:00,.@2087399d0b911ed0bb43aa9da21d7301f0a077e56c: ...,es
1422879,2017-03-21 14:55:21+00:00,Se viene la #GranMingaAgropecuaria entregaremo...,es
1422880,2017-03-21 14:55:24+00:00,Saludos amigos-as! Les comparto el mensaje sem...,es
1422881,2017-03-21 14:55:26+00:00,.@aae2514dd2b49b2e9005f27a9b7c88b4368e7b0c2d: ...,es
1422882,2017-03-21 14:55:31+00:00,En instantes siga la entrevista en vivo a @b28...,es
1422883,2017-03-21 14:55:32+00:00,Te invitamos a escuchar a @c09b5b44e08e1dee035...,es
1422884,2017-03-21 14:55:35+00:00,Como legisladores debemos fortalecer la Consti...,es
1422885,2017-03-21 14:55:38+00:00,".@c09b5b44e08e1dee0355a17be46f8d9dec3dee0d71 ""...",es



Account-history investigation completed.



**Findings:**

A conflicting post from Ecuador was associated with two different account identifiers.

The first account had 84 other posts, while the second had 83. Both accounts contained Spanish-language content, with similarities visible in their posting histories.

**Conclusion:**

The conflict may indicate an identifier inconsistency or duplicated account information. However, the evidence is insufficient to determine which account identifier is correct.

Both records are retained and flagged for further investigation. No automatic deletion or reassignment is performed.



### 3.3 Analysis of Remaining Repeated Post IDs

**Objective:** Investigate repeated country-post combinations that do not contain conflicting account identifiers.

**Methodology:**

1. Count occurrences of each `(ioCountry, postid)` combination in the cleaned dataset.
2. Identify combinations appearing more than once.
3. Exclude groups already identified as having conflicting account IDs.
4. Examine differences in post text, timestamps, control labels, and follower/following counts.
5. Summarize the remaining repeated groups.

**Purpose:**

Not all repeated records represent data-quality errors. Records may share the same post and account identifiers while containing different account statistics or other attributes.

This analysis distinguishes repeated observations with consistent identifiers from those containing additional inconsistencies.

**Cleaning Decision:**

No records are automatically removed. Remaining repeated groups are investigated before selecting an appropriate cleaning strategy.

**Outcome:** A summary of repeated country-post groups without account-ID conflicts and the attributes that differ within them.


In [21]:

# 3.3 Analysis of Remaining Repeated Post IDs


print("=== REMAINING REPEATED POST ANALYSIS ===")

group_cols = ["ioCountry", "postid"]

# Step 1: Count records per country-post combination
pair_counts = (
    df_clean.groupby(group_cols, dropna=False)
    .size()
)

# Step 2: Identify combinations occurring more than once
repeated_pairs = pair_counts[pair_counts > 1]

# Step 3: Reuse account-ID conflict groups from Section 2.5
account_conflict_keys = conflicting_groups.index[
    conflicting_groups["accountid_conflict"]
]

# Step 4: Exclude account-ID conflicts
remaining_pairs = repeated_pairs.drop(
    account_conflict_keys,
    errors="ignore"
)

# Step 5: Display summary statistics
print(f"Total repeated groups: {len(repeated_pairs):,}")
print(f"Account-ID conflict groups: {len(account_conflict_keys):,}")
print(f"Other repeated groups: {len(remaining_pairs):,}")

extra_rows = int((remaining_pairs - 1).sum())

print(f"Extra rows in other repeated groups: {extra_rows:,}")

# Step 6: Retrieve records from remaining groups
remaining_keys = remaining_pairs.index.to_frame(index=False)

remaining_records = df_clean.merge(
    remaining_keys,
    on=group_cols,
    how="inner",
    validate="many_to_one"
)

# Step 7: Compare relevant attributes
attributes = [
    "accountid",
    "post_text",
    "post_time",
    "is_control",
    "follower_count",
    "following_count"
]

remaining_differences = (
    remaining_records
    .groupby(group_cols, dropna=False)[attributes]
    .nunique(dropna=False)
    .gt(1)
    .sum()
    .sort_values(ascending=False)
)

print("\n=== ATTRIBUTES WITH DIFFERENCES ===")
display(
    remaining_differences.to_frame(
        name="Groups with different values"
    )
)

print("\n=== EXAMPLE RECORDS ===")
display(
    remaining_records[
        group_cols + attributes
    ].head(10)
)

print("\nRemaining repeated-post analysis completed.")


=== REMAINING REPEATED POST ANALYSIS ===
Total repeated groups: 296
Account-ID conflict groups: 226
Other repeated groups: 70
Extra rows in other repeated groups: 71

=== ATTRIBUTES WITH DIFFERENCES ===


,Groups with different values
post_text,70
is_control,1
follower_count,1
following_count,1
accountid,0
post_time,0



=== EXAMPLE RECORDS ===


,ioCountry,postid,accountid,post_text,post_time,is_control,follower_count,following_count
0,ecuador,a4052c913b344c3b861fffbb6346f92aa680c023b2,13a38b2ebabae523130e15d88873a54a77305173d3,me iba a despertar temprano para ver a mis nen...,2017-08-04 17:39:36+00:00,True,1005.0,577.0
1,ecuador,a4052c913b344c3b861fffbb6346f92aa680c023b2,13a38b2ebabae523130e15d88873a54a77305173d3,SAVE THE DATE \n\nViernes 18 Agosto - @aac723d...,2017-08-04 17:39:36+00:00,True,1005.0,577.0
2,ecuador,4abb0b501b26879df715d1ede5da33d85e79935dad,6b04e505d6bee92f2706dfcc66b289bd11f874c111,612 votos para mis favoritos♕\n\n {@d4a37...,2017-08-04 23:59:08+00:00,True,758.0,371.0
3,ecuador,4abb0b501b26879df715d1ede5da33d85e79935dad,6b04e505d6bee92f2706dfcc66b289bd11f874c111,611 votos para mis favoritos♕\n\n {@d4a37...,2017-08-04 23:59:08+00:00,True,758.0,371.0
4,ecuador,d3ba64d4a939d72850753e7e82685599aaa9f1e71b,58bab9718b342e0e1998c6dfbf3b48d1e6cdb40d40,cw es una joya.\n\n#CD9 #AlonsoVillalpandoTren...,2017-08-07 19:28:51+00:00,True,961.0,490.0
5,ecuador,d3ba64d4a939d72850753e7e82685599aaa9f1e71b,58bab9718b342e0e1998c6dfbf3b48d1e6cdb40d40,"Siempre que leo cw, tengo miedo pq me provoca ...",2017-08-07 19:28:51+00:00,True,961.0,490.0
6,ecuador,06590e1cdadd035275327726b0c203799f10da590a,f9a77053d5a54714ff97ee70138c3855b7268e31a6,🔥😍\n\n #KarolSevilla #MichaelRonda #ValentinaZ...,2017-08-19 16:38:43+00:00,True,964.0,1349.0
7,ecuador,06590e1cdadd035275327726b0c203799f10da590a,f9a77053d5a54714ff97ee70138c3855b7268e31a6,ENTONCES KAROL SI ESTA ALLÁ LPM #KarolSevilla ...,2017-08-19 16:38:43+00:00,True,964.0,1349.0
8,ecuador,a3560ab25b58103917ce15803dcd454f5b3800917b,fc31434046e0352ac11072e64319ec65bc4dcf23a0,A raíz del secuestro de papel por Maneiro y Cí...,2018-02-08 00:56:17+00:00,True,4114.0,4154.0
9,ecuador,a3560ab25b58103917ce15803dcd454f5b3800917b,fc31434046e0352ac11072e64319ec65bc4dcf23a0,El régimen está convocando las elecciones màs ...,2018-02-08 00:56:17+00:00,True,4114.0,4154.0



Remaining repeated-post analysis completed.



### 3.4 Validation of Remaining Repeated Records

**Objective:** Examine the remaining repeated country-post combinations to identify potentially inconsistent post content and account attributes.

**Methodology:**

1. Analyze repeated groups without account-ID conflicts.
2. Identify groups containing different post text.
3. Check whether differences also occur in `is_control`, follower counts, or following counts.
4. Distinguish isolated text conflicts from groups containing multiple conflicting attributes.

**Purpose:**

Records with identical post and account identifiers should generally contain consistent post content. Differences may indicate edited posts, collection errors, or corrupted records.

**Cleaning Decision:**

Conflicting records are retained and flagged until sufficient evidence is available to determine which observation is correct.

**Outcome:** A summary of the types of conflicts present in the remaining repeated records.


In [22]:

# 3.4 Validation of Remaining Repeated Records


print("=== REMAINING REPEATED RECORD VALIDATION ===")

group_cols = ["ioCountry", "postid"]

attributes = [
    "accountid",
    "post_text",
    "post_time",
    "is_control",
    "follower_count",
    "following_count"
]

# Step 1: Identify differences within each group
group_differences = (
    remaining_records
    .groupby(group_cols, dropna=False)[attributes]
    .nunique(dropna=False)
    .gt(1)
)

# Step 2: Classify conflict types
text_only = (
    group_differences["post_text"] &
    ~group_differences[
        ["accountid", "post_time", "is_control",
         "follower_count", "following_count"]
    ].any(axis=1)
)

multiple_conflicts = (
    group_differences.sum(axis=1) > 1
)

# Step 3: Display results
print(f"Total groups analyzed: {len(group_differences):,}")
print(f"Groups with only text differences: {text_only.sum():,}")
print(f"Groups with multiple conflicting attributes: {multiple_conflicts.sum():,}")

# Step 4: Display groups with multiple differences
print("\nGroups requiring additional investigation:")

display(
    group_differences.loc[multiple_conflicts].head(10)
)

print("\nValidation completed successfully.")


=== REMAINING REPEATED RECORD VALIDATION ===
Total groups analyzed: 70
Groups with only text differences: 69
Groups with multiple conflicting attributes: 1

Groups requiring additional investigation:


,,accountid,post_text,post_time,is_control,follower_count,following_count
ioCountry,postid,,,,,,
ecuador,1d6e232d8247d28bd95b1fc6df75cab64ba3cf2afa,False,True,False,True,True,True



Validation completed successfully.



### 3.5 Final Duplicate-Handling Decision

**Objective:** Establish a consistent strategy for handling duplicated and conflicting records.

**Summary of Findings:**

- The original dataset contains 8,714,356 records.
- 1,579 fully duplicated records were removed.
- The cleaned dataset contains 8,712,777 records.
- 296 repeated `(ioCountry, postid)` groups contain conflicts.
- 226 groups have conflicting account identifiers.
- The remaining 70 groups contain post-text differences.
- Of these 70 groups, 69 differ only in post text, while one also contains inconsistencies in `is_control`, follower count, and following count.

**Cleaning Strategy:**

1. Remove fully identical records.
2. Preserve repeated identifiers across different countries.
3. Retain records with conflicting account identifiers or post content.
4. Flag conflicting records for further investigation.
5. Avoid arbitrary deletion or reassignment when the correct observation cannot be determined.

**Final Decision:**

The 296 conflicting groups, containing 601 records, are retained and documented separately.

This approach preserves potentially useful information while avoiding unsupported cleaning decisions.

**Outcome:** A cleaned dataset without exact duplicates and a separate report identifying unresolved record-level inconsistencies.


In [23]:

# 3.5 Final Duplicate-Handling Verification

print("=== FINAL DUPLICATE-HANDLING SUMMARY ===")

original_rows = len(df)
cleaned_rows = len(df_clean)

print(f"Original records: {original_rows:,}")
print(f"Cleaned records: {cleaned_rows:,}")
print(f"Removed exact duplicates: {original_rows - cleaned_rows:,}")

print(f"Flagged conflict groups: {len(conflicting_groups):,}")

# Verify no exact duplicates remain
remaining_duplicates = df_clean.duplicated().sum()

print(f"Remaining exact duplicates: {remaining_duplicates:,}")

assert cleaned_rows == 8_712_777

# Check original duplicate removal without requiring
# further changes to the dataset
assert original_rows - cleaned_rows == 1_579

assert len(conflicting_groups) == 296

print("\nDuplicate-handling verification completed.")


=== FINAL DUPLICATE-HANDLING SUMMARY ===
Original records: 8,714,356
Cleaned records: 8,712,777
Removed exact duplicates: 1,579
Flagged conflict groups: 296
Remaining exact duplicates: 0

Duplicate-handling verification completed.



## 4. Data Consistency and Attribute Validation

### 4.1 Date Consistency Validation

**Objective:** Verify that post timestamps and account creation dates are logically consistent.

**Methodology:**

1. Check that `post_time` and `account_creation_date` contain valid datetime values.
2. Identify posts published before their associated account creation date.
3. Calculate the number and percentage of inconsistent records.
4. Inspect any invalid records before making cleaning decisions.

**Purpose:**

A post should not normally be published before the account that created it existed. Such cases may indicate incorrect timestamps or data collection errors.

**Cleaning Decision:**

Potentially inconsistent records are flagged for investigation rather than automatically removed.


In [24]:

# 4.1 Date Consistency Validation


print("=== DATE CONSISTENCY VALIDATION ===")

# Dates were already converted in Section 2.8
date_columns = ["post_time", "account_creation_date"]

# Step 1: Check missing dates
print("Missing datetime values:")
display(df_clean[date_columns].isna().sum())

# Step 2: Identify logically inconsistent dates
invalid_date_mask = (
    df_clean["post_time"].notna() &
    df_clean["account_creation_date"].notna() &
    (df_clean["post_time"] < df_clean["account_creation_date"])
)

invalid_date_records = df_clean.loc[
    invalid_date_mask,
    [
        "ioCountry",
        "postid",
        "accountid",
        "post_time",
        "account_creation_date"
    ]
].copy()

# Step 3: Display results
invalid_count = len(invalid_date_records)

print(f"\nTotal records: {len(df_clean):,}")
print(f"Posts before account creation: {invalid_count:,}")
print(f"Percentage: {invalid_count / len(df_clean) * 100:.4f}%")

# Step 4: Inspect examples
display(invalid_date_records.head(10))

print("\nDate consistency validation completed.")


=== DATE CONSISTENCY VALIDATION ===
Missing datetime values:


post_time                0
account_creation_date    0
dtype: int64


Total records: 8,712,777
Posts before account creation: 0
Percentage: 0.0000%


,ioCountry,postid,accountid,post_time,account_creation_date



Date consistency validation completed.



### 4.2 Repost Consistency Validation

**Objective:** Check whether repost indicators and referenced repost identifiers are logically consistent.

**Methodology:**

1. Examine the distribution of `is_repost`.
2. Identify reposts with missing `reposted_postid` or `reposted_accountid`.
3. Identify non-reposts containing unexpected repost identifiers.
4. Summarize potential inconsistencies for further investigation.

**Purpose:**

Repost indicators and referenced identifiers should generally agree. Inconsistencies may indicate missing information, incorrect flags, or data collection errors.

**Cleaning Decision:**

No records are automatically modified or removed. Potential inconsistencies are identified before selecting a cleaning strategy.


In [25]:

# 4.2 Repost Consistency Validation

print("=== REPOST CONSISTENCY VALIDATION ===")

# Step 1: Distribution of repost indicators
print("Repost distribution:")
display(df_clean["is_repost"].value_counts(dropna=False))

# Step 2: Check missing identifiers in reposts
repost_mask = df_clean["is_repost"].eq(True)

missing_repost_postid = (
    repost_mask &
    df_clean["reposted_postid"].isna()
)

missing_repost_accountid = (
    repost_mask &
    df_clean["reposted_accountid"].isna()
)

# Step 3: Check unexpected identifiers in non-reposts
non_repost_mask = df_clean["is_repost"].eq(False)

unexpected_repost_ids = (
    non_repost_mask &
    (
        df_clean["reposted_postid"].notna() |
        df_clean["reposted_accountid"].notna()
    )
)

# Step 4: Display results
print(f"\nReposts missing post ID: {missing_repost_postid.sum():,}")
print(f"Reposts missing account ID: {missing_repost_accountid.sum():,}")
print(f"Non-reposts with repost identifiers: {unexpected_repost_ids.sum():,}")

# Step 5: Inspect examples
problem_mask = (
    missing_repost_postid |
    missing_repost_accountid |
    unexpected_repost_ids
)

display(
    df_clean.loc[
        problem_mask,
        [
            "ioCountry",
            "postid",
            "is_repost",
            "reposted_postid",
            "reposted_accountid"
        ]
    ].head(10)
)

print("\nRepost consistency validation completed.")


=== REPOST CONSISTENCY VALIDATION ===
Repost distribution:


is_repost
False    5108257
True     3604520
Name: count, dtype: int64


Reposts missing post ID: 0
Reposts missing account ID: 0
Non-reposts with repost identifiers: 0


,ioCountry,postid,is_repost,reposted_postid,reposted_accountid



Repost consistency validation completed.



### 4.3 Reply Identifier Consistency

**Objective:** Evaluate the completeness and consistency of reply-related identifiers.

**Methodology:**

1. Count records containing `in_reply_to_postid`.
2. Count records containing `in_reply_to_accountid`.
3. Identify records where both reply identifiers are available.
4. Identify records where only one identifier is available.
5. Inspect incomplete reply relationships for possible recovery.

**Purpose:**

Reply-related identifiers should generally be consistent. A missing identifier does not necessarily represent an error because the referenced post or account may not be available in the dataset.

**Cleaning Decision:**

Incomplete reply relationships are investigated rather than automatically removed.

**Outcome:** A summary of reply identifier completeness and potential inconsistencies.


In [26]:

# 4.3 Reply Identifier Consistency


print("=== REPLY IDENTIFIER CONSISTENCY ===")

# Step 1: Check which identifiers are available
has_reply_post = df_clean["in_reply_to_postid"].notna()
has_reply_account = df_clean["in_reply_to_accountid"].notna()

# Step 2: Classify reply relationships
both_present = has_reply_post & has_reply_account
post_only = has_reply_post & ~has_reply_account
account_only = ~has_reply_post & has_reply_account
both_missing = ~has_reply_post & ~has_reply_account

# Step 3: Create summary
reply_summary = pd.DataFrame({
    "Category": [
        "Both identifiers present",
        "Reply post ID only",
        "Reply account ID only",
        "Both identifiers missing"
    ],
    "Records": [
        int(both_present.sum()),
        int(post_only.sum()),
        int(account_only.sum()),
        int(both_missing.sum())
    ]
})

reply_summary["Percentage (%)"] = (
    reply_summary["Records"] / len(df_clean) * 100
).round(2)

display(reply_summary)

# Step 4: Inspect incomplete relationships
incomplete_mask = post_only | account_only

print(f"\nIncomplete reply relationships: {incomplete_mask.sum():,}")

display(
    df_clean.loc[
        incomplete_mask,
        [
            "ioCountry",
            "postid",
            "in_reply_to_postid",
            "in_reply_to_accountid"
        ]
    ].head(10)
)

# Step 5: Verify counts
assert reply_summary["Records"].sum() == len(df_clean)

print("\nReply consistency validation completed.")


=== REPLY IDENTIFIER CONSISTENCY ===


,Category,Records,Percentage (%)
0,Both identifiers present,811470,9.31
1,Reply post ID only,74608,0.86
2,Reply account ID only,63456,0.73
3,Both identifiers missing,7763243,89.10



Incomplete reply relationships: 138,064


,ioCountry,postid,in_reply_to_postid,in_reply_to_accountid
54,ecuador,f238a72dfbc9cf111e94e7a078c4e2dc0a556d3b10,NaN,e59326834d6e1379d60784d07a4a2fe980ae08f046
112,ecuador,b341da377715b4e6070a7df209b1fd51af9ae8ed54,NaN,e5ce7a294db5ea193323597a9df3a47d9934a1bf6f
148,ecuador,33f1c596730147e4bc57bedf360942c611ed34207b,7072e5cde7a286c4a5b52b98196c9b5c24165fb33a,NaN
161,ecuador,f1ce115eb51500d7044a1c13feb7078a55648dc078,NaN,d4075bce968e013422ae6a5529bd1508fe0ff0519f
166,ecuador,153c1b6c43bd00dd07230f5030a40b1735d8e27f14,NaN,4e4621ff5197d0a57efcc55e5c50b485954e743756
234,ecuador,fce6ef3c10f7a20eec4fae362d27c02d8bccbde555,NaN,d0f04f514162246986095a5bd86d720b43986b16d0
239,ecuador,150a2f0b0683d0eddbdea2743902f3955a392332f2,8ccb92a2a1fb33f6a181eca524701c7e3e20a47876,NaN
240,ecuador,81d6449cd893ec85f3878ec93ae617732707769706,8bb358d090b579b7bab4d0c3d6733dd6f3ae132777,NaN
242,ecuador,7752a297423ca301a9b17efcab451b1b932116e547,e68036b0c506b22297941ed1f99446bdc8e8035eed,NaN
247,ecuador,70bbbd5aa334dbc6eee005db4b726d1611bae41b1b,c869f32c022a0527f8a61af1b34f1290c29a3a852c,NaN



Reply consistency validation completed.



### 4.4 Recovery of Missing Reply Account IDs

**Objective:** Recover missing `in_reply_to_accountid` values using referenced posts available in the dataset.

**Methodology:**

1. Identify records containing `in_reply_to_postid` but missing `in_reply_to_accountid`.
2. Match the referenced post ID to `postid` within the same `ioCountry`.
3. Verify that each referenced post is associated with exactly one known account ID.
4. Recover the missing account ID only when the match is unambiguous.
5. Preserve unresolved values as missing.

**Purpose:**

Reply account identifiers can sometimes be reconstructed from the authors of referenced posts.

Since anonymized identifiers may differ between countries, matches are restricted to the same country.

**Cleaning Decision:**

Only uniquely identifiable account IDs are recovered. Ambiguous and unmatched references remain missing.

**Outcome:** Improved completeness of reply relationships without introducing unsupported account assignments.


In [27]:

# 4.4 Recovery of Missing Reply Account IDs


print("=== REPLY ACCOUNT ID RECOVERY ===")

# Step 1: Identify rows requiring recovery
missing_mask = (
    df_clean["in_reply_to_postid"].notna() &
    df_clean["in_reply_to_accountid"].isna()
)

missing_before = int(missing_mask.sum())

print(f"Missing reply account IDs with known post ID: {missing_before:,}")

# Step 2: Build a reference mapping from existing posts
post_authors = (
    df_clean.dropna(subset=["accountid"])
    .groupby(["ioCountry", "postid"])["accountid"]
    .agg(["nunique", "first"])
)

# Step 3: Retain only unambiguous references
valid_authors = post_authors.loc[
    post_authors["nunique"] == 1, "first"
]

# Step 4: Match referenced post IDs within the same country
reply_keys = pd.MultiIndex.from_frame(
    df_clean.loc[
        missing_mask,
        ["ioCountry", "in_reply_to_postid"]
    ].rename(columns={
        "in_reply_to_postid": "postid"
    })
)

recovered_accounts = valid_authors.reindex(reply_keys)

# Step 5: Fill only matched missing values
target_indices = df_clean.index[missing_mask]

df_clean.loc[
    target_indices,
    "in_reply_to_accountid"
] = recovered_accounts.to_numpy()

# Step 6: Validate recovery
missing_after = int(
    (
        df_clean["in_reply_to_postid"].notna() &
        df_clean["in_reply_to_accountid"].isna()
    ).sum()
)

print(f"Recovered account IDs: {missing_before - missing_after:,}")
print(f"Remaining missing account IDs: {missing_after:,}")

assert missing_after <= missing_before
assert len(df_clean) == 8_712_777

print("\nReply account ID recovery completed.")


=== REPLY ACCOUNT ID RECOVERY ===
Missing reply account IDs with known post ID: 74,608
Recovered account IDs: 4,020
Remaining missing account IDs: 70,588

Reply account ID recovery completed.



### 4.5 Validation of Recovered Reply Identifiers

**Objective:** Verify that recovered reply account identifiers match the authors of referenced posts.

**Methodology:**

1. Identify reply relationships for which the referenced post exists within the same country.
2. Restrict validation to referenced posts with a unique associated account ID.
3. Compare available reply account IDs against the expected account IDs.
4. Count consistent and inconsistent relationships.

**Purpose:**

Successful recovery does not guarantee that every reply relationship is correct. Validation helps identify potential mismatches between reply metadata and the referenced posts.

**Cleaning Decision:**

No records are automatically modified. Inconsistent relationships are flagged for further investigation.

**Outcome:** A summary of reply identifier consistency and potentially incorrect references.


In [28]:

# 4.5 Validate Recovered Reply Identifiers


print("=== REPLY IDENTIFIER VALIDATION ===")

# Step 1: Build unique post-author mapping
post_authors = (
    df_clean.dropna(subset=["accountid"])
    .groupby(["ioCountry", "postid"])["accountid"]
    .agg(["nunique", "first"])
)

valid_authors = post_authors.loc[
    post_authors["nunique"] == 1, "first"
]

# Step 2: Select reply records with both identifiers
reply_mask = (
    df_clean["in_reply_to_postid"].notna() &
    df_clean["in_reply_to_accountid"].notna()
)

reply_rows = df_clean.loc[
    reply_mask,
    ["ioCountry", "in_reply_to_postid",
     "in_reply_to_accountid"]
].copy()

# Step 3: Match referenced posts
reply_keys = pd.MultiIndex.from_frame(
    reply_rows[
        ["ioCountry", "in_reply_to_postid"]
    ].rename(columns={
        "in_reply_to_postid": "postid"
    })
)

reply_rows["expected_accountid"] = (
    valid_authors.reindex(reply_keys).to_numpy()
)

# Step 4: Validate matched references
matched_mask = reply_rows["expected_accountid"].notna()

consistent_mask = (
    matched_mask &
    (
        reply_rows["in_reply_to_accountid"] ==
        reply_rows["expected_accountid"]
    )
)

inconsistent_mask = (
    matched_mask &
    ~consistent_mask
)

# Step 5: Display results
print(f"Replies with both identifiers: {len(reply_rows):,}")
print(f"Replies with verifiable references: {matched_mask.sum():,}")
print(f"Consistent references: {consistent_mask.sum():,}")
print(f"Inconsistent references: {inconsistent_mask.sum():,}")

print("\nExamples of inconsistencies:")
display(reply_rows.loc[inconsistent_mask].head(10))

print("\nReply identifier validation completed.")


=== REPLY IDENTIFIER VALIDATION ===
Replies with both identifiers: 815,490
Replies with verifiable references: 27,290
Consistent references: 23,842
Inconsistent references: 3,448

Examples of inconsistencies:


,ioCountry,in_reply_to_postid,in_reply_to_accountid,expected_accountid
123406,ecuador,cf158126c8ef127699b4e4deec4b879be078692dc0,5d690804d698189316db58ee2b2026549020cec597,cd0b8e047f5b20eed79cf903fd915afdf4458bc3b4
123408,ecuador,cf158126c8ef127699b4e4deec4b879be078692dc0,5d690804d698189316db58ee2b2026549020cec597,cd0b8e047f5b20eed79cf903fd915afdf4458bc3b4
123410,ecuador,cf158126c8ef127699b4e4deec4b879be078692dc0,5d690804d698189316db58ee2b2026549020cec597,cd0b8e047f5b20eed79cf903fd915afdf4458bc3b4
198162,ecuador,3423bec1aee6ed68bfe4f30a4ed0d7ff38af2ae64d,1498ae87c19763b2eec21ec2924190926938f9f23c,71cc57155d061d447c2c65756c7dfd6873e403a097
200751,ecuador,15e54495f578bf33f9617420e3118ace91e89fc2b2,f3f68db2fd6bae5034e8386845286219d831825472,7660f778af4ab8f9d4f1356263df4d15cdb4110766
294702,ecuador,ff8168244cd40a8b5e52cc3a310123781fca493831,0abf5e81703b9059f56c0364e8435ee7c8bcff02ce,a273ce9204e72b3d7798a2dd3c331f1713aa0d3111
359642,ecuador,b1a1c1d7238d2832b079a84270dd047a188b67be23,c6e536ff8d0f89c35f753212c00eca0f592b368636,52e78350374857d9b59bab84dac8a5c2b69490ed5c
361714,ecuador,d6b64105b1635eeff437e3be02021ee6e3dfd74477,9fd03bd9350439a747a5258ccf1b049cc932f1072e,b8616e238360aa0f38052b8800e05af95b0013d741
375606,ecuador,d68d950fca2790068e36ea44116c86781b8b8ee810,489f38c1abdf4d203a2df07e037d1493d7404e5721,a58b89d945727e4e70c8c7c442ea171c330e7a1801
431470,ecuador,fd128b0993c9589a725cbf5a02074fbc1d8584f357,bdf3536fd68bfc78b4591d07e4fc5bdb5f82faa6df,2fda0b815d028035df678d440c717909a7c677578b



Reply identifier validation completed.



### 4.6 Investigation of Reply Identifier Mismatches

**Objective:** Investigate inconsistencies between reply account identifiers and the authors of referenced posts.

**Methodology:**

1. Identify reply relationships where the referenced post has a unique known author.
2. Compare the recorded reply account ID with the expected author ID.
3. Determine whether mismatches existed in the original dataset or appeared after missing-value recovery.
4. Inspect examples before deciding whether corrections are appropriate.

**Purpose:**

A mismatch may indicate incorrect metadata, identifier inconsistencies, or data collection issues. Understanding its origin is necessary before modifying any records.

**Cleaning Decision:**

No identifiers are automatically replaced or removed. Unresolved inconsistencies are retained for further investigation.


In [29]:

# 4.6 Investigation of Reply Identifier Mismatches


print("=== REPLY IDENTIFIER MISMATCH INVESTIGATION ===")

# Reuse reply_rows and expected_accountid from Section 4.5
validation = reply_rows.copy()

# Retrieve original reply account IDs
validation["original_accountid"] = df.loc[
    validation.index,
    "in_reply_to_accountid"
]

# Only references with a uniquely identifiable author
verifiable = validation[
    validation["expected_accountid"].notna()
].copy()

# Identify mismatches
mismatches = verifiable[
    verifiable["in_reply_to_accountid"] !=
    verifiable["expected_accountid"]
].copy()

# Classify whether the original reply account ID was missing
mismatches["originally_missing"] = (
    mismatches["original_accountid"].isna()
)

# Display summary
print(f"Verifiable references: {len(verifiable):,}")
print(f"Total mismatches: {len(mismatches):,}")

print(
    "Mismatches already containing an account ID "
    f"in original data: "
    f"{(~mismatches['originally_missing']).sum():,}"
)

print(
    "Mismatches originally missing an account ID: "
    f"{mismatches['originally_missing'].sum():,}"
)

# Inspect examples
display(
    mismatches[
        [
            "ioCountry",
            "in_reply_to_postid",
            "original_accountid",
            "in_reply_to_accountid",
            "expected_accountid",
            "originally_missing"
        ]
    ].head(10)
)

print("\nMismatch investigation completed.")


=== REPLY IDENTIFIER MISMATCH INVESTIGATION ===
Verifiable references: 27,290
Total mismatches: 3,448
Mismatches already containing an account ID in original data: 3,448
Mismatches originally missing an account ID: 0


,ioCountry,in_reply_to_postid,original_accountid,in_reply_to_accountid,expected_accountid,originally_missing
123406,ecuador,cf158126c8ef127699b4e4deec4b879be078692dc0,5d690804d698189316db58ee2b2026549020cec597,5d690804d698189316db58ee2b2026549020cec597,cd0b8e047f5b20eed79cf903fd915afdf4458bc3b4,False
123408,ecuador,cf158126c8ef127699b4e4deec4b879be078692dc0,5d690804d698189316db58ee2b2026549020cec597,5d690804d698189316db58ee2b2026549020cec597,cd0b8e047f5b20eed79cf903fd915afdf4458bc3b4,False
123410,ecuador,cf158126c8ef127699b4e4deec4b879be078692dc0,5d690804d698189316db58ee2b2026549020cec597,5d690804d698189316db58ee2b2026549020cec597,cd0b8e047f5b20eed79cf903fd915afdf4458bc3b4,False
198162,ecuador,3423bec1aee6ed68bfe4f30a4ed0d7ff38af2ae64d,1498ae87c19763b2eec21ec2924190926938f9f23c,1498ae87c19763b2eec21ec2924190926938f9f23c,71cc57155d061d447c2c65756c7dfd6873e403a097,False
200751,ecuador,15e54495f578bf33f9617420e3118ace91e89fc2b2,f3f68db2fd6bae5034e8386845286219d831825472,f3f68db2fd6bae5034e8386845286219d831825472,7660f778af4ab8f9d4f1356263df4d15cdb4110766,False
294702,ecuador,ff8168244cd40a8b5e52cc3a310123781fca493831,0abf5e81703b9059f56c0364e8435ee7c8bcff02ce,0abf5e81703b9059f56c0364e8435ee7c8bcff02ce,a273ce9204e72b3d7798a2dd3c331f1713aa0d3111,False
359642,ecuador,b1a1c1d7238d2832b079a84270dd047a188b67be23,c6e536ff8d0f89c35f753212c00eca0f592b368636,c6e536ff8d0f89c35f753212c00eca0f592b368636,52e78350374857d9b59bab84dac8a5c2b69490ed5c,False
361714,ecuador,d6b64105b1635eeff437e3be02021ee6e3dfd74477,9fd03bd9350439a747a5258ccf1b049cc932f1072e,9fd03bd9350439a747a5258ccf1b049cc932f1072e,b8616e238360aa0f38052b8800e05af95b0013d741,False
375606,ecuador,d68d950fca2790068e36ea44116c86781b8b8ee810,489f38c1abdf4d203a2df07e037d1493d7404e5721,489f38c1abdf4d203a2df07e037d1493d7404e5721,a58b89d945727e4e70c8c7c442ea171c330e7a1801,False
431470,ecuador,fd128b0993c9589a725cbf5a02074fbc1d8584f357,bdf3536fd68bfc78b4591d07e4fc5bdb5f82faa6df,bdf3536fd68bfc78b4591d07e4fc5bdb5f82faa6df,2fda0b815d028035df678d440c717909a7c677578b,False



Mismatch investigation completed.



**Findings:**

Among 27,290 verifiable reply relationships, 23,842 were consistent and 3,448 contained mismatched account identifiers.

All 3,448 mismatches were already present in the original dataset. No mismatches were introduced by the reply account ID recovery process.

**Cleaning Decision:**

The mismatched identifiers are retained because the correct values cannot be established with sufficient confidence.

No automatic correction or deletion is performed.



### 4.7 Final Reply Consistency Summary

**Objective:** Document the results of reply identifier validation and recovery.

**Results:**

- 4,020 missing reply account IDs were successfully recovered.
- 70,588 reply account IDs with known reply post IDs remain unresolved.
- 27,290 reply references could be verified against posts in the dataset.
- 3,448 verifiable relationships contained mismatched account IDs.

**Cleaning Decision:**

Missing account IDs are recovered only when their referenced posts have a unique known author within the same country.

Existing mismatched identifiers are retained and documented separately because their correct values cannot be established with sufficient confidence.

**Outcome:** Improved reply identifier completeness and a separate report of unresolved inconsistencies.


In [30]:

# 4.7 Export Reply Identifier Conflicts

from pathlib import Path

print("=== FINAL REPLY CONSISTENCY SUMMARY ===")

output_dir = Path("../data/data_processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save inconsistent reply relationships
reply_conflicts_path = output_dir / "reply_identifier_conflicts.csv"

mismatches.to_csv(
    reply_conflicts_path,
    index=True
)

print(f"Inconsistent reply relationships: {len(mismatches):,}")
print(f"Conflict report saved to: {reply_conflicts_path}")

# Verify that the cleaned dataset was not reduced
assert len(df_clean) == 8_712_777

print("\nReply consistency analysis completed.")


=== FINAL REPLY CONSISTENCY SUMMARY ===
Inconsistent reply relationships: 3,448
Conflict report saved to: ../data/data_processed/reply_identifier_conflicts.csv

Reply consistency analysis completed.



## 5. Categorical Data Quality Assessment

### 5.1 Categorical Attribute Validation

**Objective:** Evaluate the completeness and consistency of categorical attributes.

**Methodology:**

1. Identify the categorical attributes in the dataset.
2. Count missing values for each attribute.
3. Calculate the number of unique categories.
4. Examine the most frequent categories.
5. Identify possible inconsistent or unexpected values.

**Purpose:**

Categorical inconsistencies can affect grouping, statistical analysis, and subsequent data mining algorithms.

This analysis helps identify attributes requiring further validation or cleaning.

**Cleaning Decision:**

No categorical values are automatically modified or removed during this step.

**Outcome:** A summary of categorical attributes, their completeness, and their distribution.


In [31]:

# 5.1 Categorical Attribute Validation


print("=== CATEGORICAL ATTRIBUTE VALIDATION ===")

# Step 1: Define categorical attributes
categorical_columns = [
    "ioCountry",
    "post_language",
    "application_name",
    "is_control",
    "is_repost"
]

# Step 2: Create a validation summary
results = []

for col in categorical_columns:

    values = df_clean[col]

    # Count missing values
    missing_count = values.isna().sum()

    # Count unique categories
    unique_count = values.nunique(dropna=True)

    # Calculate missing percentage
    missing_percentage = (
        missing_count / len(df_clean) * 100
    )

    results.append({
        "Attribute": col,
        "Data Type": str(values.dtype),
        "Unique Categories": unique_count,
        "Missing Values": missing_count,
        "Missing (%)": round(missing_percentage, 2)
    })

# Step 3: Display summary
categorical_summary = pd.DataFrame(results)

display(categorical_summary)

# Step 4: Inspect category distributions
print("\n=== MOST FREQUENT CATEGORIES ===")

for col in categorical_columns:

    print(f"\nAttribute: {col}")

    display(
        df_clean[col]
        .value_counts(dropna=False)
        .head(10)
    )

print("\nCategorical validation completed.")


=== CATEGORICAL ATTRIBUTE VALIDATION ===


,Attribute,Data Type,Unique Categories,Missing Values,Missing (%)
0,ioCountry,str,5,0,0.00
1,post_language,str,70,103727,1.19
2,application_name,str,3728,0,0.00
3,is_control,boolean,2,2,0.00
4,is_repost,bool,2,0,0.00



=== MOST FREQUENT CATEGORIES ===

Attribute: ioCountry


ioCountry
russia_1    3650082
uae         1707792
ecuador     1470529
iran_1      1322789
iran_4       561585
Name: count, dtype: int64


Attribute: post_language


post_language
en     2497602
ru     1568927
ar     1378222
es     1360396
fr      419150
und     373032
fa      317209
pt      134498
tr      131028
NaN     103727
Name: count, dtype: int64

... (3 outputs left out)

is_control
False    5229829
True     3482946
<NA>           2
Name: count, dtype: Int64


Attribute: is_repost


is_repost
False    5108257
True     3604520
Name: count, dtype: int64


Categorical validation completed.



### 5.2 Missing and Undetermined Language Analysis

**Objective:** Investigate missing and undetermined values in the `post_language` attribute.

**Methodology:**

1. Identify records with missing language values.
2. Identify records labeled `und` (undetermined).
3. Calculate the total number of records without a reliable language classification.
4. Examine the distribution of missing and undetermined languages across countries.
5. Inspect sample post texts to assess whether language recovery may be feasible.

**Purpose:**

Language information is important for text analysis and downstream data mining tasks.

Missing values and undetermined language labels should be distinguished because they may have different causes.

**Cleaning Decision:**

No language labels are automatically assigned. Records requiring language identification are flagged for further investigation.

**Outcome:** A summary of records requiring potential language recovery.


In [32]:

# 5.2 Missing and Undetermined Language Analysis


print("=== LANGUAGE QUALITY ANALYSIS ===")

# Step 1: Identify missing and undetermined languages
missing_language = df_clean["post_language"].isna()

undetermined_language = (
    df_clean["post_language"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("und")
    .fillna(False)
)

# Step 2: Combine categories
language_review_mask = (
    missing_language | undetermined_language
)

print(f"Missing languages: {missing_language.sum():,}")
print(f"Undetermined languages: {undetermined_language.sum():,}")
print(f"Total requiring review: {language_review_mask.sum():,}")

print(
    f"Percentage requiring review: "
    f"{language_review_mask.mean() * 100:.2f}%"
)

# Step 3: Distribution by country
print("\n=== DISTRIBUTION BY COUNTRY ===")

display(
    df_clean.loc[
        language_review_mask, "ioCountry"
    ].value_counts()
)

# Step 4: Inspect examples
print("\n=== SAMPLE POSTS ===")

display(
    df_clean.loc[
        language_review_mask,
        ["ioCountry", "post_text", "post_language"]
    ].head(15)
)

print("\nLanguage quality analysis completed.")


=== LANGUAGE QUALITY ANALYSIS ===
Missing languages: 103,727
Undetermined languages: 373,032
Total requiring review: 476,759
Percentage requiring review: 5.47%

=== DISTRIBUTION BY COUNTRY ===


ioCountry
russia_1    194756
ecuador     100022
uae          93272
iran_1       58210
iran_4       30499
Name: count, dtype: int64


=== SAMPLE POSTS ===


,ioCountry,post_text,post_language
0,ecuador,Vamos a cambiar la forma de hacer comunicación...,NaN
1,ecuador,"Damos servicio de manejo de prensa, elaboració...",NaN
2,ecuador,@db7c47cef2aa2ba68b5c010d761a03fc1c0f0d1302 Bu...,NaN
3,ecuador,"Mario, debemos denuciar que en los poderes pub...",NaN
4,ecuador,Tambien debemos estar claro que la oposición e...,NaN
5,ecuador,Otro problema que se esta haciendo en bajo per...,NaN
6,ecuador,Familiarizandome con el Twitter...,NaN
7,ecuador,Alguien me puede decir por que el Twitter es t...,NaN
8,ecuador,Brindamos servicios integrales de relaciones p...,NaN
9,ecuador,Contactenos: 094914378... [EMAIL],NaN



Language quality analysis completed.



### 5.3 Language Detection Feasibility Assessment

**Objective:** Determine which posts with missing or undetermined language labels contain sufficient textual information for language detection.

**Methodology:**

1. Identify records where `post_language` is missing or labeled `und`.
2. Calculate post-text lengths.
3. Classify posts according to their textual length.
4. Identify posts containing only URLs, mentions, or non-linguistic content.
5. Estimate how many records are suitable for automatic language detection.

**Purpose:**

Language detection may be unreliable for very short texts or posts containing little meaningful language.

Evaluating text quality before applying language detection helps reduce unnecessary processing and unreliable predictions.

**Cleaning Decision:**

No language values are modified during this assessment. Language detection will be applied only after reviewing the eligibility results.


In [33]:

# 5.3 Language Detection Feasibility Assessment


print("=== LANGUAGE DETECTION FEASIBILITY ===")

# Step 1: Identify missing or undetermined languages
language_values = (
    df_clean["post_language"]
    .astype("string")
    .str.strip()
    .str.lower()
)

review_mask = (
    language_values.isna() |
    language_values.eq("und").fillna(False)
)

# Step 2: Select only records requiring review
language_candidates = df_clean.loc[
    review_mask,
    ["ioCountry", "post_text", "post_language"]
].copy()

# Step 3: Calculate text length
text = language_candidates["post_text"].fillna("").astype(str)

language_candidates["text_length"] = text.str.len()

# Remove URLs, mentions and hashtags for diagnostic purposes
clean_text = (
    text
    .str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
    .str.replace(r"@\w+", " ", regex=True)
    .str.replace(r"#", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Count alphabetic characters, including non-Latin scripts
language_candidates["letter_count"] = clean_text.str.count(
    r"[^\W\d_]"
)

# Step 4: Classify candidates using diagnostic thresholds
language_candidates["detection_category"] = "Needs review"

language_candidates.loc[
    language_candidates["letter_count"] < 5,
    "detection_category"
] = "Insufficient text"

language_candidates.loc[
    language_candidates["letter_count"] >= 20,
    "detection_category"
] = "Suitable candidate"

# Step 5: Display summary
summary = (
    language_candidates["detection_category"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="Records")
)

summary["Percentage (%)"] = (
    summary["Records"] / len(language_candidates) * 100
).round(2)

display(summary)

print(f"\nTotal candidates: {len(language_candidates):,}")

print("\nText length statistics:")
display(language_candidates["text_length"].describe())

print("\nSample candidates:")
display(language_candidates.head(10))

print("\nFeasibility assessment completed.")


=== LANGUAGE DETECTION FEASIBILITY ===


,Category,Records,Percentage (%)
0,Suitable candidate,225825,47.37
1,Insufficient text,173962,36.49
2,Needs review,76972,16.14



Total candidates: 476,759

Text length statistics:


count    476759.000000
mean        101.932249
std         121.213156
min           1.000000
25%          50.000000
50%          75.000000
75%         118.000000
max        2368.000000
Name: text_length, dtype: float64


Sample candidates:


,ioCountry,post_text,post_language,text_length,letter_count,detection_category
0,ecuador,Vamos a cambiar la forma de hacer comunicación...,NaN,58,50,Suitable candidate
1,ecuador,"Damos servicio de manejo de prensa, elaboració...",NaN,133,108,Suitable candidate
2,ecuador,@db7c47cef2aa2ba68b5c010d761a03fc1c0f0d1302 Bu...,NaN,136,75,Suitable candidate
3,ecuador,"Mario, debemos denuciar que en los poderes pub...",NaN,139,115,Suitable candidate
4,ecuador,Tambien debemos estar claro que la oposición e...,NaN,139,117,Suitable candidate
5,ecuador,Otro problema que se esta haciendo en bajo per...,NaN,138,113,Suitable candidate
6,ecuador,Familiarizandome con el Twitter...,NaN,34,28,Suitable candidate
7,ecuador,Alguien me puede decir por que el Twitter es t...,NaN,65,47,Suitable candidate
8,ecuador,Brindamos servicios integrales de relaciones p...,NaN,139,119,Suitable candidate
9,ecuador,Contactenos: 094914378... [EMAIL],NaN,33,16,Needs review



Feasibility assessment completed.



### 5.4 Language Recovery Using Existing Post Text

**Objective:** Determine whether missing and undetermined language labels can be recovered from identical texts already present in the dataset.

**Methodology:**

1. Identify posts with valid existing language labels.
2. Group identical post texts.
3. Retain text groups associated with exactly one known language.
4. Match missing or undetermined language records against these groups.
5. Recover language labels only when the mapping is unambiguous.

**Purpose:**

Repeated post content may already have a reliable language label in another observation.

Reusing existing labels reduces the number of posts requiring external language detection.

**Cleaning Decision:**

Only unambiguous matches are used. Conflicting language labels and unmatched texts remain unchanged.

**Outcome:** A summary of language labels recoverable from existing dataset information.


In [34]:

# 5.4 Language Recovery Using Existing Post Text


print("=== EXISTING TEXT LANGUAGE RECOVERY ===")

language = (
    df_clean["post_language"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Identify missing or undetermined languages
missing_mask = (
    language.isna() |
    language.eq("und").fillna(False)
)

# Valid reference labels
valid_mask = (
    ~missing_mask &
    df_clean["post_text"].notna()
)

# Group identical texts with known languages
known_texts = df_clean.loc[
    valid_mask,
    ["post_text"]
].copy()

known_texts["language"] = language.loc[valid_mask]

# Retain texts with one unique known language
text_languages = (
    known_texts.groupby("post_text")["language"]
    .agg(["nunique", "first"])
)

unique_text_languages = text_languages.loc[
    text_languages["nunique"] == 1,
    "first"
]

# Match records requiring recovery
candidate_texts = df_clean.loc[
    missing_mask, "post_text"
]

recovered_languages = candidate_texts.map(
    unique_text_languages
)

# Calculate recovery potential
recoverable = recovered_languages.notna().sum()

print(f"Records requiring review: {missing_mask.sum():,}")
print(f"Recoverable using identical text: {recoverable:,}")
print(f"Still requiring detection: {missing_mask.sum() - recoverable:,}")

# Apply only unambiguous recovery
matched_indices = recovered_languages.index[
    recovered_languages.notna()
]

df_clean.loc[
    matched_indices,
    "post_language"
] = recovered_languages.loc[matched_indices].to_numpy()

print("\nExisting-text language recovery completed.")


=== EXISTING TEXT LANGUAGE RECOVERY ===
Records requiring review: 476,759
Recoverable using identical text: 2,193
Still requiring detection: 474,566

Existing-text language recovery completed.



## 6. Final Data Quality Validation and Export


### 6.1 Cleaning and Recovery of Negative Numeric Values

**Objective:** Identify, correct, and recover invalid negative values in numerical attributes.

**Problem:**

The final data-quality investigation identified:
- 977 negative values in `follower_count`.
- 1,029 negative values in `following_count`.

Follower and following counts cannot be negative.

**Methodology:**

1. Identify negative values in both numerical attributes.
2. Replace invalid negative values with `NaN`.
3. Find valid observations belonging to the same `(ioCountry, accountid)` combination.
4. Recover missing values using the nearest valid observation in time.
5. Restrict recovery to a maximum time difference of 30 days.
6. Preserve unresolved values as missing rather than assigning arbitrary values.
7. Verify that no negative values remain.

**Purpose:** Improve numerical consistency while preserving the original records.

**Recovery limitation:** The nearest observation is an estimate, not necessarily the true historical value. The 30-day tolerance is a conservative project assumption.

**Expected outcome:** No negative follower or following counts, with a documented recovery summary.



In [35]:

# ============================================================
# 6.1 CLEANING AND RECOVERY OF NEGATIVE NUMERIC VALUES
# ============================================================

numeric_columns = ["follower_count", "following_count"]
MAX_TIME_GAP = pd.Timedelta(days=30)

print("=" * 60)
print("6.1 NEGATIVE NUMERIC VALUE CLEANING & RECOVERY")
print("=" * 60)

recovery_summary = []

for col in numeric_columns:

    # Step 1: Identify invalid negative values
    negative_mask = df_clean[col] < 0
    negative_count = int(negative_mask.sum())

    # Step 2: Replace negative values with NaN
    df_clean.loc[negative_mask, col] = float("nan")

    # Step 3: Select valid observations
    valid = df_clean.loc[
        df_clean[col].notna() & df_clean[col].ge(0),
        ["ioCountry", "accountid", "post_time", col]
    ].copy()

    # Recover only the originally negative observations
    missing = df_clean.loc[
        negative_mask,
        ["ioCountry", "accountid", "post_time"]
    ].copy()

    missing["_row_id"] = missing.index

    recovered_count = 0

    if not missing.empty:

        # Step 4: Sort chronologically
        valid = valid.sort_values("post_time")
        missing = missing.sort_values("post_time")

        # Step 5: Match nearest valid observation
        recovered = pd.merge_asof(
            missing,
            valid,
            on="post_time",
            by=["ioCountry", "accountid"],
            direction="nearest",
            tolerance=MAX_TIME_GAP
        )

        # Step 6: Apply successful recoveries
        successful = recovered[col].notna()

        df_clean.loc[
            recovered.loc[successful, "_row_id"],
            col
        ] = recovered.loc[successful, col].to_numpy()

        recovered_count = int(successful.sum())

    # Step 7: Verify results
    remaining_negative = int((df_clean[col] < 0).sum())
    remaining_missing = int(df_clean[col].isna().sum())

    recovery_summary.append({
        "Attribute": col,
        "Negative Found": negative_count,
        "Recovered": recovered_count,
        "Unresolved Negative Cases": (
            negative_count - recovered_count
        ),
        "Total Missing": remaining_missing,
        "Remaining Negative": remaining_negative
    })

    print(f"\n{col}:")
    print(f"Negative values found: {negative_count:,}")
    print(f"Recovered: {recovered_count:,}")
    print(f"Remaining missing: {remaining_missing:,}")
    print(f"Remaining negative: {remaining_negative:,}")

# Final report
print("\n=== RECOVERY SUMMARY ===")
recovery_summary_df = pd.DataFrame(recovery_summary)
display(recovery_summary_df)

assert all(
    (df_clean[col] >= 0).all()
    if df_clean[col].notna().all()
    else (df_clean[col].dropna() >= 0).all()
    for col in numeric_columns
), "Negative values remain!"

print("\nNegative-value cleaning completed.")


6.1 NEGATIVE NUMERIC VALUE CLEANING & RECOVERY

follower_count:
Negative values found: 977
Recovered: 974
Remaining missing: 5
Remaining negative: 0

following_count:
Negative values found: 1,029
Recovered: 1,029
Remaining missing: 3
Remaining negative: 0

=== RECOVERY SUMMARY ===


,Attribute,Negative Found,Recovered,Unresolved Negative Cases,Total Missing,Remaining Negative
0,follower_count,977,974,3,5,0
1,following_count,1029,1029,0,3,0



Negative-value cleaning completed.



### 6.3 Final Cleaned Dataset Export

**Objective:** Export the cleaned and validated dataset for subsequent data mining activities.

**Methodology:**

1. Verify that all final integrity checks passed.
2. Save the cleaned dataset as a compressed CSV file.
3. Preserve the original 20 attributes and all cleaned records.
4. Confirm that the exported file exists and is not empty.
5. Record the final dataset dimensions and file size.

**Data preservation policy:**

- Retain records with documented identifier conflicts.
- Preserve unresolved language labels (`NaN` and `und`).
- Preserve missing values when reliable recovery is not possible.
- Do not introduce additional transformations during export.

**Purpose:** Provide a consistent, reproducible dataset for Part 3 (Exploratory Data Analysis) and subsequent clustering.

**Expected outcome:** A compressed CSV file containing the final validated dataset.



### 6.2 Final Dataset Integrity Validation

**Objective:** Perform a comprehensive validation of the cleaned dataset before exporting it for subsequent data mining tasks.

**Methodology:**

1. Verify dataset dimensions and original attributes.
2. Check for remaining exact duplicate records.
3. Summarize missing values across all attributes.
4. Identify unresolved language labels, including `NaN` and `und`.
5. Validate follower and following counts for negative, fractional, and extreme values.
6. Verify numerical, datetime, and Boolean data types.
7. Check the chronological consistency of posts and account creation dates.
8. Validate repost identifiers against repost status.
9. Produce a final quality report with PASS/FAIL results.

**Data-quality decisions:**

- Unresolved language labels are retained pending further evaluation.
- Missing reply/repost identifiers are not automatically invalid because they may not apply to the corresponding posts.
- Previously identified ambiguous records are retained and documented.
- No additional values are modified during this validation.

**Purpose:** Verify the integrity of the cleaned dataset and document any remaining issues before export.

**Expected outcome:** A complete quality report confirming whether the cleaned dataset satisfies the defined integrity checks.


In [37]:

# ============================================================
# 6.2 FINAL DATASET INTEGRITY VALIDATION
# ============================================================
# This cell validates data without modifying df_clean.

print("=" * 60)
print("6.2 FINAL DATA QUALITY VALIDATION")
print("=" * 60)

quality_checks = []


def add_check(name, passed, details):
    quality_checks.append({
        "Check": name,
        "Status": "PASS" if passed else "FAIL",
        "Details": str(details)
    })


# ------------------------------------------------------------
# 1. DATASET STRUCTURE
# ------------------------------------------------------------

print("\n[1] DATASET STRUCTURE")

total_rows, total_columns = df_clean.shape

print(f"Total rows: {total_rows:,}")
print(f"Total columns: {total_columns}")

add_check(
    "Expected row count",
    total_rows == 8_712_777,
    f"{total_rows:,} rows"
)

add_check(
    "Expected column count",
    total_columns == 20,
    f"{total_columns} columns"
)

add_check(
    "Original columns preserved",
    set(df_clean.columns) == set(df.columns)
    and df_clean.columns.is_unique,
    "Original 20 attributes"
)

add_check(
    "Unique DataFrame index",
    df_clean.index.is_unique,
    "Index validation"
)


# ------------------------------------------------------------
# 2. EXACT DUPLICATES
# ------------------------------------------------------------

print("\n[2] DUPLICATE VALIDATION")

duplicates = int(df_clean.duplicated().sum())

print(f"Exact duplicates: {duplicates:,}")

add_check(
    "Exact duplicate records",
    duplicates == 0,
    f"{duplicates:,} duplicates"
)


# ------------------------------------------------------------
# 3. MISSING VALUES
# ------------------------------------------------------------

print("\n[3] MISSING VALUES")

missing_summary = pd.DataFrame({
    "Missing Count": df_clean.isna().sum(),
    "Missing (%)": (
        df_clean.isna().mean() * 100
    ).round(2)
}).sort_values("Missing Count", ascending=False)

display(missing_summary)

essential_columns = [
    "ioCountry",
    "postid",
    "accountid",
    "post_time",
    "account_creation_date",
    "is_repost"
]

for col in essential_columns:
    count = int(df_clean[col].isna().sum())

    add_check(
        f"Missing: {col}",
        count == 0,
        f"{count:,} missing"
    )

# Language values requiring review
language_missing = int(
    df_clean["post_language"].isna().sum()
)

language_und = int(
    df_clean["post_language"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("und")
    .sum()
)

print("\nLanguage labels requiring review:")
print(f"NaN: {language_missing:,}")
print(f"und: {language_und:,}")
print(
    f"Total unresolved: "
    f"{language_missing + language_und:,}"
)


# ------------------------------------------------------------
# 4. NUMERIC VALIDATION
# ------------------------------------------------------------

print("\n[4] NUMERIC VALIDATION")

numeric_results = []

for col in ["follower_count", "following_count"]:

    values = pd.to_numeric(
        df_clean[col],
        errors="coerce"
    )

    missing_count = int(values.isna().sum())
    negative_count = int((values < 0).sum())

    fractional_count = int(
        (values.notna() & (values % 1 != 0)).sum()
    )

    extreme_count = int(
        (values > 1_000_000_000).sum()
    )

    numeric_results.append({
        "Attribute": col,
        "Missing": missing_count,
        "Negative": negative_count,
        "Fractional": fractional_count,
        "Above 1 Billion": extreme_count
    })

    add_check(
        f"Non-negative: {col}",
        negative_count == 0,
        f"{negative_count:,} negative"
    )

    add_check(
        f"Integer values: {col}",
        fractional_count == 0,
        f"{fractional_count:,} fractional"
    )

    add_check(
        f"Extreme values: {col}",
        extreme_count == 0,
        f"{extreme_count:,} extreme"
    )

display(pd.DataFrame(numeric_results))


# ------------------------------------------------------------
# 5. DATA TYPE VALIDATION
# ------------------------------------------------------------

print("\n[5] DATA TYPES")

important_columns = [
    "follower_count",
    "following_count",
    "post_time",
    "account_creation_date",
    "is_control",
    "is_repost"
]

display(df_clean[important_columns].dtypes)

for col in ["post_time", "account_creation_date"]:

    valid = pd.api.types.is_datetime64_any_dtype(
        df_clean[col]
    )

    add_check(
        f"Datetime: {col}",
        valid,
        str(df_clean[col].dtype)
    )

for col in ["is_control", "is_repost"]:

    valid = pd.api.types.is_bool_dtype(
        df_clean[col]
    )

    add_check(
        f"Boolean: {col}",
        valid,
        str(df_clean[col].dtype)
    )

for col in ["follower_count", "following_count"]:

    valid = pd.api.types.is_numeric_dtype(
        df_clean[col]
    )

    add_check(
        f"Numeric: {col}",
        valid,
        str(df_clean[col].dtype)
    )


# ------------------------------------------------------------
# 6. DATE CONSISTENCY
# ------------------------------------------------------------

print("\n[6] DATE CONSISTENCY")

invalid_dates = int(
    (
        df_clean["post_time"]
        < df_clean["account_creation_date"]
    ).sum()
)

print(f"Invalid chronological records: {invalid_dates:,}")

add_check(
    "Post/account date consistency",
    invalid_dates == 0,
    f"{invalid_dates:,} inconsistent records"
)


# ------------------------------------------------------------
# 7. REPOST CONSISTENCY
# ------------------------------------------------------------

print("\n[7] REPOST CONSISTENCY")

is_repost = df_clean["is_repost"]

missing_repost_ids = int(
    (
        is_repost.eq(True).fillna(False)
        & (
            df_clean["reposted_postid"].isna()
            | df_clean["reposted_accountid"].isna()
        )
    ).sum()
)

unexpected_repost_ids = int(
    (
        is_repost.eq(False).fillna(False)
        & (
            df_clean["reposted_postid"].notna()
            | df_clean["reposted_accountid"].notna()
        )
    ).sum()
)

print(f"Reposts without IDs: {missing_repost_ids:,}")
print(f"Non-reposts with IDs: {unexpected_repost_ids:,}")

add_check(
    "Reposts have identifiers",
    missing_repost_ids == 0,
    f"{missing_repost_ids:,} inconsistent"
)

add_check(
    "Non-reposts lack identifiers",
    unexpected_repost_ids == 0,
    f"{unexpected_repost_ids:,} inconsistent"
)


# ------------------------------------------------------------
# 8. FINAL QUALITY REPORT
# ------------------------------------------------------------

print("\n[8] FINAL QUALITY REPORT")

quality_report = pd.DataFrame(quality_checks)

display(quality_report)

passed = int(quality_report["Status"].eq("PASS").sum())
failed = int(quality_report["Status"].eq("FAIL").sum())

print("\n" + "=" * 60)
print("FINAL VALIDATION SUMMARY")
print("=" * 60)

print(f"Total checks: {len(quality_report)}")
print(f"Passed checks: {passed}")
print(f"Failed checks: {failed}")

if failed == 0:
    print("\nSTATUS: ALL DEFINED CHECKS PASSED")
    print("Dataset is ready for final export.")
else:
    print("\nSTATUS: ISSUES REQUIRE REVIEW")

    display(
        quality_report.loc[
            quality_report["Status"] == "FAIL"
        ]
    )


6.2 FINAL DATA QUALITY VALIDATION

[1] DATASET STRUCTURE
Total rows: 8,712,777
Total columns: 20

[2] DUPLICATE VALIDATION
Exact duplicates: 0

[3] MISSING VALUES


,Missing Count,Missing (%)
in_reply_to_accountid,7833831,89.91
in_reply_to_postid,7826699,89.83
reposted_postid,5108257,58.63
reposted_accountid,5108257,58.63
account_profile_description,1669612,19.16
post_language,101757,1.17
follower_count,5,0.00
post_text,3,0.00
following_count,3,0.00
is_control,2,0.00



Language labels requiring review:
NaN: 101,757
und: 372,809
Total unresolved: 474,566

[4] NUMERIC VALIDATION


,Attribute,Missing,Negative,Fractional,Above 1 Billion
0,follower_count,5,0,0,0
1,following_count,3,0,0,0



[5] DATA TYPES


follower_count                       float64
following_count                      float64
post_time                datetime64[us, UTC]
account_creation_date    datetime64[us, UTC]
is_control                           boolean
is_repost                               bool
dtype: object


[6] DATE CONSISTENCY
Invalid chronological records: 0

[7] REPOST CONSISTENCY
Reposts without IDs: 0
Non-reposts with IDs: 0

[8] FINAL QUALITY REPORT


,Check,Status,Details
0,Expected row count,PASS,"8,712,777 rows"
1,Expected column count,PASS,20 columns
2,Original columns preserved,PASS,Original 20 attributes
3,Unique DataFrame index,PASS,Index validation
4,Exact duplicate records,PASS,0 duplicates
5,Missing: ioCountry,PASS,0 missing
6,Missing: postid,PASS,0 missing
7,Missing: accountid,PASS,0 missing
8,Missing: post_time,PASS,0 missing
9,Missing: account_creation_date,PASS,0 missing



FINAL VALIDATION SUMMARY
Total checks: 26
Passed checks: 26
Failed checks: 0

STATUS: ALL DEFINED CHECKS PASSED
Dataset is ready for final export.


In [38]:

# ============================================================
# 6.3 FINAL CLEANED DATASET EXPORT
# ============================================================

from pathlib import Path

print("=" * 60)
print("6.3 FINAL CLEANED DATASET EXPORT")
print("=" * 60)

# ------------------------------------------------------------
# 1. VERIFY FINAL VALIDATION
# ------------------------------------------------------------

assert "quality_report" in globals(), (
    "Run Section 6.2 before exporting the dataset."
)

assert (quality_report["Status"] == "PASS").all(), (
    "Final validation contains failed checks."
)

assert len(df_clean) == 8_712_777, (
    "Unexpected number of records."
)

assert df_clean.shape[1] == 20, (
    "Unexpected number of attributes."
)

print("\nFinal validation: PASSED")


# ------------------------------------------------------------
# 2. DEFINE OUTPUT PATH
# ------------------------------------------------------------

output_dir = Path("../data/data_processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "df_clean_final.csv.gz"

print(f"\nOutput file: {output_path.resolve()}")


# ------------------------------------------------------------
# 3. EXPORT CLEANED DATASET
# ------------------------------------------------------------

print("\nExporting cleaned dataset...")

df_clean.to_csv(
    output_path,
    index=False,
    compression="gzip"
)


# ------------------------------------------------------------
# 4. VERIFY EXPORTED FILE
# ------------------------------------------------------------

assert output_path.exists(), (
    "Export failed: file does not exist."
)

file_size_mb = output_path.stat().st_size / (1024 ** 2)

assert file_size_mb > 0, (
    "Export failed: file is empty."
)

# Verify CSV header without loading millions of rows
exported_columns = pd.read_csv(
    output_path,
    nrows=0
).columns.tolist()

assert exported_columns == df_clean.columns.tolist(), (
    "Exported columns do not match the cleaned dataset."
)


# ------------------------------------------------------------
# 5. FINAL EXPORT SUMMARY
# ------------------------------------------------------------

export_summary = pd.DataFrame({
    "Metric": [
        "Total records",
        "Total attributes",
        "Validation checks passed",
        "Validation checks failed",
        "Compressed file size (MB)",
        "Export status"
    ],
    "Result": [
        f"{len(df_clean):,}",
        df_clean.shape[1],
        int(quality_report["Status"].eq("PASS").sum()),
        int(quality_report["Status"].eq("FAIL").sum()),
        round(file_size_mb, 2),
        "SUCCESS"
    ]
})

print("\n=== FINAL EXPORT SUMMARY ===")
display(export_summary)

print("\nDataset exported successfully.")
print(f"Saved to: {output_path.resolve()}")


6.3 FINAL CLEANED DATASET EXPORT

Final validation: PASSED

Output file: /Users/ramy/Courses/Data Mining /project/Data-Mining-Project-/data/data_processed/df_clean_final.csv.gz

Exporting cleaned dataset...

=== FINAL EXPORT SUMMARY ===


,Metric,Result
0,Total records,"8,712,777"
1,Total attributes,20
2,Validation checks passed,26
3,Validation checks failed,0
4,Compressed file size (MB),1278.02
5,Export status,SUCCESS



Dataset exported successfully.
Saved to: /Users/ramy/Courses/Data Mining /project/Data-Mining-Project-/data/data_processed/df_clean_final.csv.gz
